# Predicting Crash Severity for Proactive Safety Resource Allocation in Chicago
**ML Foundations — Midterm Project**

**Goal:** Predict whether a traffic crash will result in severe outcomes and translate spatial risk patterns into actionable policy recommendations for CDOT resource allocation.

**Team:** Rizaldy Utomo, Utami, Diyouva Novith  
Carnegie Mellon University, MSPPM — Data Analytics

---

### Section Directory

- [0. Setup](#0-setup)
- [1. Data Preparation](#1-data-preparation)
- [2. Feature Engineering & EDA](#2-feature-engineering--eda)
- [3. Classification](#3-classification)
- [4. Spatial Clustering](#4-spatial-clustering)
- [5. Spatial Dashboard & Policy Integration](#5-spatial-dashboard--policy-integration)
- [6. Key Takeaways & Policy Recommendations](#6-key-takeaways--policy-recommendations)


## Executive Summary

- The pipeline cleans and splits the Chicago crash dataset into leakage-safe train and test sets, then engineers interpretable temporal, spatial, roadway, and behavioral features for modeling.
- Exploratory analysis shows severe crashes are rare, vary across time and context, and remain spatially concentrated enough to support location-prioritized intervention design.
- Crash-level classification and condition-level risk scoring are combined: notebook 03 trains supervised models and exports `risk_scores.csv` for downstream geographic prioritization.
- Spatial clustering groups grid cells into risk neighborhoods, and notebook 05 turns those outputs into a dashboard-ready set of maps, figures, and policy tables.
- This unified notebook preserves the team’s original notebook logic and interpretation cells while writing consolidated figures to `image/main/`.


In [ ]:
# 0) Setup
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from folium import plugins
from urllib.error import URLError
from urllib.request import urlopen

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, average_precision_score,
    silhouette_score, silhouette_samples
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
MAIN_IMAGE_DIR = Path('image/main') if Path('image').exists() else Path('../image/main')
MAIN_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT.resolve())
print('Main image dir:', MAIN_IMAGE_DIR.resolve())


## 1) Data Preparation

## Alignment Notes
- Core question: How do we build a clean, leakage-aware temporal dataset that is ready for modeling and policy analysis?
- Predicting task in this notebook: None. This notebook prepares inputs for later predictive notebooks.
- Based on what data: Historical Chicago crash records from `dataset/training_data_2023_2025.csv`.
- Unit of analysis: Individual crash record.
- Main process: quality checks, missing-value handling, target construction (`is_severe`), and temporal train/test split.
- Output artifacts:
  - `dataset/cleaned_data/train.csv`
  - `dataset/cleaned_data/test.csv`
  - `dataset/cleaned_data/feature_columns.json`
  - `dataset/cleaned_data/data_prep_metadata.json`
- Answers for team: Provides the canonical base dataset and split used by notebooks 02, 03, 04, and 05.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


## 1) Load and validate source data


In [ ]:
DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
RAW_PATH = DATA_ROOT / 'training_data_2023_2025.csv'
assert RAW_PATH.exists(), f'Missing input file: {RAW_PATH.resolve()}'

df = pd.read_csv(RAW_PATH, low_memory=False)
df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'], errors='coerce')

print('Source:', RAW_PATH.resolve())
print('Shape:', df.shape)
print('Columns:', len(df.columns))
print('Date range:', df['CRASH_DATE'].min(), 'to', df['CRASH_DATE'].max())


## 2) Column audit and null-rate review


In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
null_rate = df.isna().mean().sort_values(ascending=False)
print('Top 15 columns by null rate (%):')
print((null_rate.head(15) * 100).round(2).astype(str) + '%')

high_null_cols = null_rate[null_rate > 0.70].index.tolist()
print('\nColumns above 70% null:', len(high_null_cols))
print(high_null_cols)


## 3) Cleaning and data-quality handling


In [ ]:
# Drop columns with >70% null
if high_null_cols:
    df = df.drop(columns=high_null_cols)

# Normalize binary Y/blank columns where present
for col in ['INTERSECTION_RELATED_I', 'HIT_AND_RUN_I']:
    if col in df.columns:
        df[col] = (
            df[col]
            .fillna('N')
            .astype(str)
            .str.strip()
            .str.upper()
            .replace({'': 'N'})
        )

# Numeric cleanup
for col in ['POSTED_SPEED_LIMIT', 'NUM_UNITS', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'LATITUDE', 'LONGITUDE']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['POSTED_SPEED_LIMIT', 'NUM_UNITS', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Fill categorical missing as UNKNOWN
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN').astype(str).str.strip().replace({'': 'UNKNOWN'})

# Geocode cleanup
before_geo = len(df)
df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])
df = df[(df['LATITUDE'] != 0) & (df['LONGITUDE'] != 0)]
print('Rows removed by invalid/missing geocode:', before_geo - len(df))

# Deduplicate
if 'CRASH_RECORD_ID' in df.columns:
    before_dup = len(df)
    df = df.drop_duplicates(subset=['CRASH_RECORD_ID'])
    print('Duplicates removed:', before_dup - len(df))

# Keep self-report signal
if 'REPORT_TYPE' in df.columns:
    df['is_self_reported'] = df['REPORT_TYPE'].str.upper().str.contains('NOT ON SCENE').astype(int)
else:
    df['is_self_reported'] = 0


## 4) Target creation and leakage-safe base columns


In [ ]:
for col in ['INJURIES_FATAL', 'INJURIES_INCAPACITATING']:
    assert col in df.columns, f'Missing required target column: {col}'
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['is_severe'] = ((df['INJURIES_FATAL'] > 0) | (df['INJURIES_INCAPACITATING'] > 0)).astype(int)

target_counts = df['is_severe'].value_counts().sort_index()
target_pct = (target_counts / target_counts.sum() * 100).round(2)
print('Class distribution:')
print(target_counts)
print(target_pct)

# Keep base columns for feature engineering + EDA in notebook 02
# Diyouva-compatible base contract retains NUM_UNITS, DAMAGE, PRIM_CONTRIBUTORY_CAUSE,
# and INTERSECTION_RELATED_I so notebook 02 can rebuild the intended feature matrix.
base_cols = [
    'CRASH_RECORD_ID', 'CRASH_DATE',
    'POSTED_SPEED_LIMIT', 'NUM_UNITS', 'DAMAGE',
    'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION',
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE',
    'TRAFFICWAY_TYPE', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT',
    'PRIM_CONTRIBUTORY_CAUSE',
    'INTERSECTION_RELATED_I', 'HIT_AND_RUN_I', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
    'LATITUDE', 'LONGITUDE', 'REPORT_TYPE', 'is_self_reported', 'is_severe'
]

base_cols = [c for c in base_cols if c in df.columns]
df_base = df[base_cols].copy()
df_base = df_base.dropna(subset=['CRASH_DATE']).sort_values('CRASH_DATE').reset_index(drop=True)

print('Base dataset shape:', df_base.shape)
print('Base columns:', len(df_base.columns))


## 5) Practical temporal split (about 75/25)


In [ ]:
split_idx = int(len(df_base) * 0.75)
split_date = pd.to_datetime(df_base.loc[split_idx, 'CRASH_DATE']).floor('D')

train = df_base[df_base['CRASH_DATE'] < split_date].copy()
test = df_base[df_base['CRASH_DATE'] >= split_date].copy()


def summarize(name, frame):
    print(f"{name}: rows={len(frame):,}, severe_rate={(frame['is_severe'].mean() * 100):.2f}%")
    print('  date range:', frame['CRASH_DATE'].min(), 'to', frame['CRASH_DATE'].max())


print('Split date:', split_date)
summarize('Train', train)
summarize('Test', test)
print('Train share:', round(len(train) / len(df_base), 4))
print('Test share:', round(len(test) / len(df_base), 4))


## 6) Save outputs and enforce max file size


In [ ]:
OUT_DIR = DATA_ROOT / 'cleaned_data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_path = OUT_DIR / 'train.csv'
test_path = OUT_DIR / 'test.csv'
meta_path = OUT_DIR / 'data_prep_metadata.json'

train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)

meta = {
    'target': 'is_severe',
    'split_strategy': 'time-based ~75/25',
    'split_date': str(split_date.date()),
    'n_train': int(len(train)),
    'n_test': int(len(test)),
    'train_share': float(len(train) / len(df_base)),
    'test_share': float(len(test) / len(df_base)),
}
meta_path.write_text(json.dumps(meta, indent=2))

max_mb = 100
for p in [train_path, test_path]:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'{p.name}: {size_mb:.2f} MB')
    assert size_mb < max_mb, f'{p.name} is {size_mb:.2f} MB, exceeds {max_mb} MB'

print('Saved:')
print('-', train_path.resolve())
print('-', test_path.resolve())
print('-', meta_path.resolve())


# Base feature metadata for team alignment
feature_meta_path = OUT_DIR / 'feature_columns.json'
feature_cols = [
    c for c in train.columns
    if c not in ['is_severe', 'CRASH_DATE', 'CRASH_RECORD_ID']
]
feature_meta = {
    'target': 'is_severe',
    'date_col': 'CRASH_DATE',
    'base_feature_cols': feature_cols,
    'n_features': len(feature_cols),
}
feature_meta_path.write_text(json.dumps(feature_meta, indent=2))
print('-', feature_meta_path.resolve())


## 2) Feature Engineering & EDA

## Alignment Notes
- Core question: Which crash conditions and spatial-temporal patterns are associated with severe outcomes before formal modeling?
- Predicting task in this notebook: None. This notebook performs feature engineering and diagnostic EDA.
- Based on what data: Base train/test outputs from notebook 01 (`train.csv`, `test.csv`).
- Unit of analysis:
  - Crash-level rows for feature diagnostics.
  - Aggregated condition profiles (`grid_cell x weather x lighting x time_bucket`) for risk-preview analysis.
- Main process: grouped feature engineering, class/temporal/categorical/numerical/spatial EDA, and aggregation preview.
- Output artifacts:
  - `dataset/cleaned_data/train_model.csv`
  - `dataset/cleaned_data/test_model.csv`
  - `dataset/cleaned_data/aggregation_preview_top_profiles.csv`
  - EDA figures in `image/02_eda/`
- Answers for team: Defines interpretable features and evidence that motivate both crash-level and condition-level modeling in notebook 03.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
TRAIN_PATH = DATA_ROOT / 'cleaned_data' / 'train.csv'
TEST_PATH = DATA_ROOT / 'cleaned_data' / 'test.csv'
assert TRAIN_PATH.exists() and TEST_PATH.exists(), 'Run 01_data_preparation.ipynb first.'

IMAGE_DIR = MAIN_IMAGE_DIR
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(filename: str, dpi: int = 180) -> Path:
    out = IMAGE_DIR / filename
    plt.savefig(out, dpi=dpi, bbox_inches='tight')
    print('Saved figure:', out.resolve())
    return out


train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
for frame in [train_raw, test_raw]:
    frame['CRASH_DATE'] = pd.to_datetime(frame['CRASH_DATE'], errors='coerce')

print('Train base shape:', train_raw.shape)
print('Test base shape:', test_raw.shape)
print('Train severe rate:', round(train_raw['is_severe'].mean() * 100, 2), '%')
print('Test severe rate:', round(test_raw['is_severe'].mean() * 100, 2), '%')
print('Image output folder:', IMAGE_DIR.resolve())

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

## 0) Feature engineering for modeling


### Group A: Incident-context binary features

Justification: intersection crashes, hit-and-run events, and self-reported records, and also some traffic way type  are operationally meaningful signals for severity risk.


In [ ]:
def normalize_text(series: pd.Series) -> pd.Series:
    return series.fillna('UNKNOWN').astype(str).str.upper().str.strip().replace({'': 'UNKNOWN'})

df_fe = pd.concat([
    train_raw.assign(_split='train'),
    test_raw.assign(_split='test')
], ignore_index=True)

for col in ['INTERSECTION_RELATED_I', 'HIT_AND_RUN_I']:
    if col in df_fe.columns:
        df_fe[col] = (
            df_fe[col]
            .fillna('N')
            .astype(str)
            .str.strip()
            .str.upper()
            .replace({'': 'N'})
        )

if 'INTERSECTION_RELATED_I' in df_fe.columns:
    df_fe['is_intersection'] = (df_fe['INTERSECTION_RELATED_I'] == 'Y').astype(int)
else:
    df_fe['is_intersection'] = 0

if 'HIT_AND_RUN_I' in df_fe.columns:
    df_fe['is_hit_and_run'] = (df_fe['HIT_AND_RUN_I'] == 'Y').astype(int)
else:
    df_fe['is_hit_and_run'] = 0

if 'is_self_reported' not in df_fe.columns and 'REPORT_TYPE' in df_fe.columns:
    df_fe['is_self_reported'] = df_fe['REPORT_TYPE'].astype(str).str.upper().str.contains('NOT ON SCENE').astype(int)

In [ ]:
# Severe crash risk by roadway structure

road_env = (
    train_raw.groupby('TRAFFICWAY_TYPE', as_index=False)
            .agg(
                crash_count=('is_severe','size'),
                severe_count=('is_severe','sum'),
                severe_rate=('is_severe','mean')
            )
)

road_env = road_env[road_env['crash_count'] >= 50]   # remove unstable groups
road_env['severe_rate_pct'] = road_env['severe_rate'] * 100

road_env = road_env.sort_values('severe_rate_pct')

plt.figure(figsize=(10,6))

bars = plt.barh(
    road_env['TRAFFICWAY_TYPE'],
    road_env['severe_rate_pct'],
    color='#4C72B0'
)

plt.xlabel('Severe crash probability (%)')
plt.ylabel('Trafficway type')
plt.title('Severe Crash Risk by Roadway Type')

for bar, n in zip(bars, road_env['crash_count']):
    plt.text(
        bar.get_width()+0.05,
        bar.get_y()+bar.get_height()/2,
        f'n={n}',
        va='center'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Binary trafficway indicators for EDA interpretation
if 'TRAFFICWAY_TYPE' in df_fe.columns:
    t = normalize_text(df_fe['TRAFFICWAY_TYPE'])
    df_fe['is_four_way'] = (t == 'FOUR WAY').astype(int)
    df_fe['is_five_point'] = t.str.contains('FIVE POINT', na=False).astype(int)
    df_fe['is_y_intersection'] = t.str.contains('Y-INTERSECTION', na=False).astype(int)
    df_fe['is_t_intersection'] = t.str.contains('T-INTERSECTION', na=False).astype(int)
    df_fe['is_divided_road'] = t.str.contains('DIVIDED', na=False).astype(int)
else:
    df_fe['is_four_way'] = 0
    df_fe['is_five_point'] = 0
    df_fe['is_y_intersection'] = 0
    df_fe['is_t_intersection'] = 0
    df_fe['is_divided_road'] = 0


### Group B: Temporal risk features

Justification: Time patterns strongly influence crash severity. Time windows such as nighttime, weekend, and rush hour are actionable for policy targeting and operational deployment. Chicago crash severity may differ in winter vs summer.


In [ ]:
for col in ['CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']:
    if col in df_fe.columns:
        df_fe[col] = pd.to_numeric(df_fe[col], errors='coerce')

if 'CRASH_HOUR' in df_fe.columns:
    df_fe['is_night'] = df_fe['CRASH_HOUR'].isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
    df_fe['is_rush_hour'] = df_fe['CRASH_HOUR'].isin([7, 8, 9, 16, 17, 18]).astype(int)
else:
    df_fe['is_night'] = 0
    df_fe['is_rush_hour'] = 0

if 'CRASH_DAY_OF_WEEK' in df_fe.columns:
    df_fe['is_weekend'] = df_fe['CRASH_DAY_OF_WEEK'].isin([1, 7]).astype(int)
else:
    df_fe['is_weekend'] = 0

# Season 
def get_season(month):
    if pd.isna(month):
        return 'UNKNOWN'
    month = int(month)
    if month in [12, 1, 2]:
        return 'WINTER'
    elif month in [3, 4, 5]:
        return 'SPRING'
    elif month in [6, 7, 8]:
        return 'SUMMER'
    elif month in [9, 10, 11]:
        return 'FALL'
    return 'UNKNOWN'

df_fe['season'] = df_fe['CRASH_MONTH'].apply(get_season)

In [ ]:
# Weather and visibility signals
if 'WEATHER_CONDITION' in df_fe.columns:
    weather = normalize_text(df_fe['WEATHER_CONDITION'])

    weather_standardized = np.select(
        [
            weather.str.contains('BLOWING SNOW'),
            weather.str.contains('CLEAR'),
            weather.str.contains('CLOUD'),
            weather.str.contains('FOG|SMOKE|HAZE'),
            weather.str.contains('FREEZING RAIN|FREEZING DRIZZLE'),
            weather.str.contains('RAIN|DRIZZLE'),
            weather.str.contains('SLEET|HAIL'),
            weather.str.contains('SNOW'),
            weather.eq('UNKNOWN')
        ],
        [
            'BLOWING SNOW', 'CLEAR', 'CLOUDY/OVERCAST', 'FOG/SMOKE/HAZE',
            'FREEZING RAIN/DRIZZLE', 'RAIN', 'SLEET/HAIL', 'SNOW', 'UNKNOWN'
        ],
        default='OTHER'
    )
    df_fe['WEATHER_CONDITION'] = weather_standardized

    df_fe['weather_bucket'] = np.select(
        [
            weather.str.contains('CLEAR'),
            weather.str.contains('CLOUD'),
            weather.str.contains('RAIN|DRIZZLE'),
            weather.str.contains('SNOW|SLEET|ICE|FREEZING|BLOWING SNOW'),
            weather.str.contains('FOG|SMOKE|HAZE'),
            weather.eq('UNKNOWN')
        ],
        ['CLEAR', 'CLOUDY', 'RAIN', 'SNOW_ICE', 'LOW_VISIBILITY', 'UNKNOWN'],
        default='OTHER'
    )

    df_fe['adverse_weather'] = df_fe['weather_bucket'].isin(
        ['RAIN', 'SNOW_ICE', 'LOW_VISIBILITY']
    ).astype(int)
else:
    df_fe['WEATHER_CONDITION'] = 'UNKNOWN'
    df_fe['weather_bucket'] = 'UNKNOWN'
    df_fe['adverse_weather'] = 0

if 'LIGHTING_CONDITION' in df_fe.columns:
    lighting = normalize_text(df_fe['LIGHTING_CONDITION'])
    df_fe['is_dark'] = lighting.isin([
        'DARKNESS', 'DARKNESS, LIGHTED ROAD', 'DAWN', 'DUSK'
    ]).astype(int)
else:
    df_fe['is_dark'] = 0

df_fe['low_visibility'] = (
    (df_fe['adverse_weather'] == 1) |
    (df_fe['is_dark'] == 1)
).astype(int)

if 'ROADWAY_SURFACE_COND' in df_fe.columns:
    surface = normalize_text(df_fe['ROADWAY_SURFACE_COND'])
    df_fe['slippery_surface'] = surface.isin([
        'WET', 'SNOW OR SLUSH', 'ICE', 'SAND, MUD, DIRT'
    ]).astype(int)
else:
    df_fe['slippery_surface'] = 0

if 'POSTED_SPEED_LIMIT' in df_fe.columns:
    speed = pd.to_numeric(df_fe['POSTED_SPEED_LIMIT'], errors='coerce')
    df_fe['speed_tier'] = pd.cut(
        speed,
        bins=[-np.inf, 20, 30, 40, np.inf],
        labels=['LOW', 'MEDIUM', 'HIGH', 'VERY_HIGH']
    ).astype(str)
    df_fe['high_speed_area'] = (speed >= 35).astype(int)
else:
    df_fe['speed_tier'] = 'UNKNOWN'
    df_fe['high_speed_area'] = 0

df_fe['night_weekend'] = (
    (df_fe['is_night'] == 1) &
    (df_fe['is_weekend'] == 1)
).astype(int)

df_fe['dark_high_speed'] = (
    (df_fe['is_dark'] == 1) &
    (df_fe['high_speed_area'] == 1)
).astype(int)

df_fe['low_visibility_high_speed'] = (
    (df_fe['low_visibility'] == 1) &
    (df_fe['high_speed_area'] == 1)
).astype(int)


### Group C: Grouped high-cardinality features

Justification: grouping high-cardinality categories reduces sparsity and variance, while preserving useful semantic structure for model stability.


In [ ]:
if 'TRAFFIC_CONTROL_DEVICE' in df_fe.columns:
    tcd = normalize_text(df_fe['TRAFFIC_CONTROL_DEVICE'])
    df_fe['traffic_control_group'] = np.select(
        [
            tcd.str.contains('SIGNAL'),
            tcd.str.contains('STOP'),
            tcd.str.contains('NO CONTROLS|NONE|UNKNOWN'),
        ],
        ['SIGNAL', 'STOP_SIGN', 'NO_CONTROLS_OR_UNKNOWN'],
        default='OTHER',
    )

if 'FIRST_CRASH_TYPE' in df_fe.columns:
    fct = normalize_text(df_fe['FIRST_CRASH_TYPE'])
    df_fe['first_crash_group'] = np.select(
        [
            fct.eq('ANGLE'),
            fct.eq('FIXED OBJECT'),
            fct.eq('PARKED MOTOR VEHICLE'),
            fct.eq('PEDESTRIAN'),
            fct.eq('REAR END'),
            fct.eq('SIDESWIPE SAME DIRECTION'),
            fct.eq('TURNING'),
        ],
        [
            'ANGLE', 'FIXED OBJECT', 'PARKED MOTOR VEHICLE', 'PEDESTRIAN',
            'REAR END', 'SIDESWIPE SAME DIRECTION', 'TURNING'
        ],
        default='OTHER'
    )

    df_fe['crash_type_risk_group'] = np.select(
        [
            fct.str.contains('PEDESTRIAN|PEDALCYCLE'),
            fct.str.contains('HEAD ON'),
            fct.str.contains('ANGLE'),
            fct.str.contains('REAR END'),
            fct.str.contains('FIXED OBJECT'),
            fct.str.contains('SIDESWIPE'),
            fct.str.contains('TURNING'),
        ],
        [
            'PEDESTRIAN_CYCLIST', 'HEAD_ON', 'ANGLE', 'REAR_END',
            'FIXED_OBJECT', 'SIDESWIPE', 'TURNING'
        ],
        default='OTHER'
    )

if 'PRIM_CONTRIBUTORY_CAUSE' in df_fe.columns:
    cause = normalize_text(df_fe['PRIM_CONTRIBUTORY_CAUSE'])
    priority_causes = {
        'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE',
        'FAILING TO REDUCE SPEED TO AVOID CRASH',
        'FAILING TO YIELD RIGHT-OF-WAY',
        'FOLLOWING TOO CLOSELY',
        'IMPROPER BACKING',
        'IMPROPER LANE USAGE',
        'IMPROPER OVERTAKING/PASSING',
        'IMPROPER TURNING/NO SIGNAL',
        'NOT APPLICABLE',
        'UNABLE TO DETERMINE',
    }
    df_fe['prim_cause_group'] = np.where(cause.isin(priority_causes), cause, 'OTHER')

# EDA helper copies for interpretable plots
for col in [
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'ROADWAY_SURFACE_COND',
    'traffic_control_group', 'crash_type_risk_group'
]:
    if col in df_fe.columns:
        df_fe[f'eda_{col.lower()}'] = df_fe[col].astype(str)

def trafficway_structure_label(row):
    if row['is_five_point'] == 1:
        return 'FIVE POINT OR MORE'
    if row['is_four_way'] == 1:
        return 'FOUR WAY'
    if row['is_y_intersection'] == 1:
        return 'Y-INTERSECTION'
    if row['is_t_intersection'] == 1:
        return 'T-INTERSECTION'
    if row['is_divided_road'] == 1:
        return 'DIVIDED ROAD'
    return 'OTHER'

df_fe['eda_trafficway_structure'] = df_fe.apply(trafficway_structure_label, axis=1)


### Group D: One-hot encoding and compact model matrix

Justification: one-hot encoding is appropriate for low-cardinality categories. Ultra-rare dummy columns are removed to reduce noise and keep exported CSV sizes practical.


In [ ]:
one_hot_cols = [
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'ROADWAY_SURFACE_COND',
    'ALIGNMENT', 'DEVICE_CONDITION', 'ROAD_DEFECT', 'DAMAGE', 'TRAFFICWAY_TYPE',
    'traffic_control_group', 'first_crash_group', 'prim_cause_group'
]

one_hot_cols = [c for c in one_hot_cols if c in df_fe.columns]

model_df = pd.get_dummies(df_fe, columns=one_hot_cols, prefix_sep='__', dtype=int)

# Remove identifier/text fields from model matrix
# Keep Diyouva-compatible numeric fields and raw columns only when they are encoded above.
drop_for_model = [
    'CRASH_RECORD_ID', 'REPORT_TYPE', 'INTERSECTION_RELATED_I', 'HIT_AND_RUN_I',
    'TRAFFIC_CONTROL_DEVICE', 'FIRST_CRASH_TYPE', 'PRIM_CONTRIBUTORY_CAUSE',
    'weather_bucket', 'speed_tier',
    'crash_type_risk_group',
    'is_four_way', 'is_five_point', 'is_y_intersection', 'is_t_intersection', 'is_divided_road',
    'adverse_weather', 'is_dark', 'low_visibility', 'slippery_surface',
    'high_speed_area', 'night_weekend', 'dark_high_speed', 'low_visibility_high_speed',
    'season'
]

# Remove EDA-only helper copies from modeling matrix
eda_helper_cols = [col for col in model_df.columns if col.startswith('eda_')]
if eda_helper_cols:
    model_df = model_df.drop(columns=eda_helper_cols)
for col in drop_for_model:
    if col in model_df.columns:
        model_df = model_df.drop(columns=col)

for col in model_df.columns:
    if col in ['CRASH_DATE', '_split']:
        continue
    if model_df[col].dtype == 'O':
        model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

for col in model_df.columns:
    if col in ['CRASH_DATE', '_split']:
        continue
    if pd.api.types.is_numeric_dtype(model_df[col]) and model_df[col].isna().any():
        model_df[col] = model_df[col].fillna(model_df[col].median())

# Align to Diyouva's intended feature contract while keeping notebook execution reproducible.
required_feature_cols = [
    "POSTED_SPEED_LIMIT",
    "NUM_UNITS",
    "CRASH_HOUR",
    "CRASH_DAY_OF_WEEK",
    "CRASH_MONTH",
    "LATITUDE",
    "LONGITUDE",
    "is_self_reported",
    "is_intersection",
    "is_hit_and_run",
    "is_night",
    "is_rush_hour",
    "is_weekend",
    "WEATHER_CONDITION__BLOWING SNOW",
    "WEATHER_CONDITION__CLEAR",
    "WEATHER_CONDITION__CLOUDY/OVERCAST",
    "WEATHER_CONDITION__FOG/SMOKE/HAZE",
    "WEATHER_CONDITION__FREEZING RAIN/DRIZZLE",
    "WEATHER_CONDITION__OTHER",
    "WEATHER_CONDITION__RAIN",
    "WEATHER_CONDITION__SLEET/HAIL",
    "WEATHER_CONDITION__SNOW",
    "WEATHER_CONDITION__UNKNOWN",
    "LIGHTING_CONDITION__DARKNESS",
    "LIGHTING_CONDITION__DARKNESS, LIGHTED ROAD",
    "LIGHTING_CONDITION__DAWN",
    "LIGHTING_CONDITION__DAYLIGHT",
    "LIGHTING_CONDITION__DUSK",
    "LIGHTING_CONDITION__UNKNOWN",
    "ROADWAY_SURFACE_COND__DRY",
    "ROADWAY_SURFACE_COND__ICE",
    "ROADWAY_SURFACE_COND__OTHER",
    "ROADWAY_SURFACE_COND__SNOW OR SLUSH",
    "ROADWAY_SURFACE_COND__UNKNOWN",
    "ROADWAY_SURFACE_COND__WET",
    "ALIGNMENT__CURVE ON GRADE",
    "ALIGNMENT__CURVE, LEVEL",
    "ALIGNMENT__STRAIGHT AND LEVEL",
    "ALIGNMENT__STRAIGHT ON GRADE",
    "ALIGNMENT__STRAIGHT ON HILLCREST",
    "DEVICE_CONDITION__FUNCTIONING IMPROPERLY",
    "DEVICE_CONDITION__FUNCTIONING PROPERLY",
    "DEVICE_CONDITION__NO CONTROLS",
    "DEVICE_CONDITION__NOT FUNCTIONING",
    "DEVICE_CONDITION__OTHER",
    "DEVICE_CONDITION__UNKNOWN",
    "ROAD_DEFECT__DEBRIS ON ROADWAY",
    "ROAD_DEFECT__NO DEFECTS",
    "ROAD_DEFECT__OTHER",
    "ROAD_DEFECT__RUT, HOLES",
    "ROAD_DEFECT__SHOULDER DEFECT",
    "ROAD_DEFECT__UNKNOWN",
    "ROAD_DEFECT__WORN SURFACE",
    "DAMAGE__$500 OR LESS",
    "DAMAGE__$501 - $1,500",
    "DAMAGE__OVER $1,500",
    "TRAFFICWAY_TYPE__ALLEY",
    "TRAFFICWAY_TYPE__CENTER TURN LANE",
    "TRAFFICWAY_TYPE__DIVIDED - W/MEDIAN (NOT RAISED)",
    "TRAFFICWAY_TYPE__DIVIDED - W/MEDIAN BARRIER",
    "TRAFFICWAY_TYPE__DRIVEWAY",
    "TRAFFICWAY_TYPE__FIVE POINT, OR MORE",
    "TRAFFICWAY_TYPE__FOUR WAY",
    "TRAFFICWAY_TYPE__NOT DIVIDED",
    "TRAFFICWAY_TYPE__NOT REPORTED",
    "TRAFFICWAY_TYPE__ONE-WAY",
    "TRAFFICWAY_TYPE__OTHER",
    "TRAFFICWAY_TYPE__PARKING LOT",
    "TRAFFICWAY_TYPE__RAMP",
    "TRAFFICWAY_TYPE__ROUNDABOUT",
    "TRAFFICWAY_TYPE__T-INTERSECTION",
    "TRAFFICWAY_TYPE__TRAFFIC ROUTE",
    "TRAFFICWAY_TYPE__UNKNOWN",
    "TRAFFICWAY_TYPE__UNKNOWN INTERSECTION TYPE",
    "TRAFFICWAY_TYPE__Y-INTERSECTION",
    "traffic_control_group__NO_CONTROLS_OR_UNKNOWN",
    "traffic_control_group__OTHER",
    "traffic_control_group__SIGNAL",
    "traffic_control_group__STOP_SIGN",
    "first_crash_group__ANGLE",
    "first_crash_group__FIXED OBJECT",
    "first_crash_group__OTHER",
    "first_crash_group__PARKED MOTOR VEHICLE",
    "first_crash_group__PEDESTRIAN",
    "first_crash_group__REAR END",
    "first_crash_group__SIDESWIPE SAME DIRECTION",
    "first_crash_group__TURNING",
    "prim_cause_group__DRIVING SKILLS/KNOWLEDGE/EXPERIENCE",
    "prim_cause_group__FAILING TO REDUCE SPEED TO AVOID CRASH",
    "prim_cause_group__FAILING TO YIELD RIGHT-OF-WAY",
    "prim_cause_group__FOLLOWING TOO CLOSELY",
    "prim_cause_group__IMPROPER BACKING",
    "prim_cause_group__IMPROPER LANE USAGE",
    "prim_cause_group__IMPROPER OVERTAKING/PASSING",
    "prim_cause_group__IMPROPER TURNING/NO SIGNAL",
    "prim_cause_group__NOT APPLICABLE",
    "prim_cause_group__OTHER",
    "prim_cause_group__UNABLE TO DETERMINE"
]

for col in required_feature_cols:
    if col not in model_df.columns:
        model_df[col] = 0

# Remove columns outside the Diyouva contract except the target/date/split contract fields.
keep_cols = set(required_feature_cols) | {'is_severe', 'CRASH_DATE', '_split'}
extra_cols = [c for c in model_df.columns if c not in keep_cols]
if extra_cols:
    model_df = model_df.drop(columns=extra_cols)

train_model = model_df[model_df['_split'] == 'train'].drop(columns=['_split']).copy()
test_model = model_df[model_df['_split'] == 'test'].drop(columns=['_split']).copy()

feature_cols = required_feature_cols.copy()

OUT_DIR = DATA_ROOT / 'cleaned_data'
train_model_path = OUT_DIR / 'train_model.csv'
test_model_path = OUT_DIR / 'test_model.csv'
feature_meta_path = OUT_DIR / 'feature_columns.json'

train_model.to_csv(train_model_path, index=False)
test_model.to_csv(test_model_path, index=False)
feature_meta_path.write_text(json.dumps({
    'feature_cols': feature_cols,
    'target': 'is_severe',
    'date_col': 'CRASH_DATE',
    'model_train_file': 'train_model.csv',
    'model_test_file': 'test_model.csv'
}, indent=2))

for p in [train_model_path, test_model_path]:
    print(f"{p.name}: {p.stat().st_size / (1024 * 1024):.2f} MB")

print('Model feature count:', len(feature_cols))


## 1) Target distribution


In [ ]:
target_counts = train_raw['is_severe'].value_counts().sort_index()
target_pct = (target_counts / target_counts.sum() * 100).round(2)

plot_df = pd.DataFrame({
    'label': ['Non-severe (0)', 'Severe (1)'],
    'count': [target_counts.get(0, 0), target_counts.get(1, 0)],
})

plt.figure(figsize=(8, 5))
bars = plt.bar(plot_df['label'], plot_df['count'], color=['#4C72B0', '#C44E52'])
plt.title('Crash Severity Class Distribution (Train Set)')
plt.xlabel('Class')
plt.ylabel('Crash count')
for bar, count in zip(bars, plot_df['count']):
    plt.text(bar.get_x() + bar.get_width() / 2, count, f'{int(count):,}', ha='center', va='bottom')
save_fig('01_target_distribution.png')
plt.show()

print('Class percentages:')
print(target_pct)


Interpretation:
- The severe class is a small minority, so imbalance-aware evaluation is necessary.
- In downstream modeling, prioritize recall-sensitive diagnostics alongside precision.


## 2) Temporal patterns


In [ ]:
monthly = (
    train_raw.assign(month=train_raw['CRASH_DATE'].dt.to_period('M').dt.to_timestamp())
             .groupby('month', as_index=False)
             .agg(crash_count=('is_severe', 'size'), severe_rate=('is_severe', 'mean'))
)
monthly['severe_rate_pct'] = monthly['severe_rate'] * 100

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(monthly['month'], monthly['crash_count'], color='#4C72B0', marker='o')
ax1.set_ylabel('Crash count', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')
ax1.set_title('Monthly Crash Volume and Severe Rate (Train)')

ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['severe_rate_pct'], color='#C44E52', marker='o')
ax2.set_ylabel('Severe rate (%)', color='#C44E52')
ax2.tick_params(axis='y', labelcolor='#C44E52')

fig.autofmt_xdate()
save_fig('02_monthly_crash_and_severity.png')
plt.show()


In [ ]:
hourly = train_raw.groupby('CRASH_HOUR', as_index=False)['is_severe'].mean()
hourly['severe_rate_pct'] = hourly['is_severe'] * 100

dow = train_raw.groupby('CRASH_DAY_OF_WEEK', as_index=False)['is_severe'].mean()
dow['severe_rate_pct'] = dow['is_severe'] * 100
day_map = {1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed', 5: 'Thu', 6: 'Fri', 7: 'Sat'}
dow['day_name'] = dow['CRASH_DAY_OF_WEEK'].map(day_map)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=hourly, x='CRASH_HOUR', y='severe_rate_pct', color='#55A868', ax=axes[0])
axes[0].set_title('Severe Crash Rate by Hour')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Severe rate (%)')

sns.barplot(data=dow, x='day_name', y='severe_rate_pct', color='#8172B2', ax=axes[1], order=['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat'])
axes[1].set_title('Severe Crash Rate by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Severe rate (%)')

plt.tight_layout()
save_fig('03_severe_rate_hour_day.png')
plt.show()


In [ ]:
heat = (
    train_raw.pivot_table(index='CRASH_DAY_OF_WEEK', columns='CRASH_HOUR', values='is_severe', aggfunc='mean') * 100
)
heat = heat.reindex(index=[1, 2, 3, 4, 5, 6, 7])
heat.index = [day_map.get(i, str(i)) for i in heat.index]

plt.figure(figsize=(14, 5))
sns.heatmap(heat, cmap='YlOrRd', cbar_kws={'label': 'Severe rate (%)'})
plt.title('Hour x Day-of-Week Severe Crash Rate Heatmap')
plt.xlabel('Hour')
plt.ylabel('Day of week')
save_fig('04_hour_day_heatmap.png')
plt.show()


In [ ]:
top_hours = hourly.nlargest(3, 'severe_rate_pct')[['CRASH_HOUR', 'severe_rate_pct']]
top_days = dow.nlargest(3, 'severe_rate_pct')[['day_name', 'severe_rate_pct']]
top_months = monthly.nlargest(3, 'severe_rate_pct')[['month', 'severe_rate_pct']]

print('Temporal hotspot summary:')
print('Top 3 hours by severe rate (%)')
print(top_hours.to_string(index=False))
print('\nTop 3 days by severe rate (%)')
print(top_days.to_string(index=False))
print('\nTop 3 months by severe rate (%)')
print(top_months.to_string(index=False))


## 3) Categorical features


In [ ]:
cat_specs = [
    ('Weather condition', ['eda_weather_condition', 'WEATHER_CONDITION'], '08_cat_weather_condition.png'),
    ('Lighting condition', ['eda_lighting_condition', 'LIGHTING_CONDITION'], '09_cat_lighting_condition.png'),
    ('Roadway surface', ['eda_roadway_surface_cond', 'ROADWAY_SURFACE_COND'], '10_cat_roadway_surface.png'),
    ('Crash type risk group', ['eda_crash_type_risk_group', 'crash_type_risk_group', 'FIRST_CRASH_TYPE'], '11_cat_crash_type_risk_group.png'),
    ('Traffic control', ['eda_traffic_control_group', 'traffic_control_group', 'TRAFFIC_CONTROL_DEVICE'], '12_cat_traffic_control.png'),
    ('Trafficway structure', ['eda_trafficway_structure'], '13_cat_trafficway_structure.png')
]

train_fe = df_fe[df_fe['_split'] == 'train'].copy()


def resolve_column(candidates):
    for c in candidates:
        if c in train_fe.columns:
            return c
    return None


def severe_rate_by_category(frame, col, min_count=200, top_n=12):
    tmp = frame[[col, 'is_severe']].copy()
    tmp[col] = tmp[col].astype(str).str.strip().replace({'': 'UNKNOWN'}).fillna('UNKNOWN')
    stats = tmp.groupby(col)['is_severe'].agg(severe_rate='mean', n='size').reset_index()
    filtered = stats[stats['n'] >= min_count].copy()
    if filtered.empty:
        filtered = stats
    filtered['rate_pct'] = filtered['severe_rate'] * 100
    return filtered.sort_values(['rate_pct', 'n'], ascending=[False, False]).head(top_n)


def plot_category_rates(stats, col, title, filename):
    plot_df = stats.sort_values(['rate_pct', 'n'], ascending=[True, True]).reset_index(drop=True)
    plt.figure(figsize=(11, max(5, len(plot_df) * 0.42)))
    ax = sns.barplot(data=plot_df, x='rate_pct', y=col, color='#4C72B0')
    ax.set_title(title)
    ax.set_xlabel('Severe crash rate (%)')
    ax.set_ylabel('Category')
    m = plot_df['rate_pct'].max() if len(plot_df) else 0
    for i, row in plot_df.iterrows():
        ax.text(row['rate_pct'] + max(0.03, m * 0.01), i, f"n={int(row['n'])}", va='center', fontsize=9)
    save_fig(filename)
    plt.show()


cat_highlights = {}
for title, candidates, filename in cat_specs:
    col = resolve_column(candidates)
    if col is None:
        print(f'Skipped {title}: column not found.')
        continue
    stats = severe_rate_by_category(train_fe, col)
    if stats.empty:
        continue
    plot_category_rates(stats, col, f'{title}: severe rate by category', filename)
    top = stats.iloc[0]
    cat_highlights[title] = {'category': str(top[col]), 'rate_pct': float(top['rate_pct']), 'n': int(top['n'])}

print('Categorical highlights:')
for k, v in cat_highlights.items():
    print(f"- {k}: {v['category']} ({v['rate_pct']:.2f}%, n={v['n']})")


Interpretation:
- Severe-risk rates differ materially across categorical groups, supporting grouped categorical features.
- Category-level conclusions should be read together with sample sizes to avoid over-interpreting sparse groups.


## 4) Numerical features


In [ ]:
speed_stats = {}

if 'POSTED_SPEED_LIMIT' in train_fe.columns:
    tmp = train_fe[['POSTED_SPEED_LIMIT', 'is_severe']].copy()
    tmp['POSTED_SPEED_LIMIT'] = pd.to_numeric(tmp['POSTED_SPEED_LIMIT'], errors='coerce')
    tmp = tmp.dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.histplot(tmp['POSTED_SPEED_LIMIT'], bins=25, color='#55A868', ax=axes[0])
    axes[0].set_title('Posted Speed Limit Distribution')
    axes[0].set_xlabel('Posted speed limit')

    sns.boxplot(data=tmp, x='is_severe', y='POSTED_SPEED_LIMIT', ax=axes[1], color='#4C72B0')
    axes[1].set_title('Speed Limit by Severity')
    axes[1].set_xlabel('is_severe')
    axes[1].set_xticks([0, 1], ['0', '1'])

    plt.tight_layout()
    save_fig('14_speed_distribution_and_boxplot.png')
    plt.show()

    speed_stats = {
        'median_non_severe': float(tmp.loc[tmp['is_severe'] == 0, 'POSTED_SPEED_LIMIT'].median()),
        'median_severe': float(tmp.loc[tmp['is_severe'] == 1, 'POSTED_SPEED_LIMIT'].median()),
    }


In [ ]:
corr_candidates = [
    'POSTED_SPEED_LIMIT', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
    'is_self_reported', 'is_hit_and_run', 'is_intersection', 'is_weekend', 'is_night',
    'is_rush_hour', 'LATITUDE', 'LONGITUDE', 'is_severe',   'is_four_way', 'is_five_point', 'is_y_intersection', 'is_t_intersection', 'is_divided_road'
]
corr_cols = [c for c in corr_candidates if c in train_fe.columns]
corr_df = train_fe[corr_cols].copy()
for col in corr_df.columns:
    corr_df[col] = pd.to_numeric(corr_df[col], errors='coerce')

corr = corr_df.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, square=True)
plt.title('Correlation Heatmap (Selected Numeric Features)')
save_fig('16_numeric_correlation_heatmap.png')
plt.show()

if 'is_severe' in corr.columns:
    _corr_series = corr['is_severe'].drop('is_severe').dropna()
    target_corr = _corr_series.reindex(_corr_series.abs().sort_values(ascending=False).index)
    print('Top absolute correlations with is_severe:')
    print(target_corr.head(8).to_string())


## 5) Spatial analysis


In [ ]:
spatial = train_raw.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()
print('Rows with valid coordinates:', len(spatial))

sample_n = min(50000, len(spatial))
spatial_sample = spatial.sample(sample_n, random_state=42)
severe_sample = spatial_sample[spatial_sample['is_severe'] == 1]

plt.figure(figsize=(10, 8))
plt.scatter(spatial_sample['LONGITUDE'], spatial_sample['LATITUDE'], s=2, alpha=0.15, label='All crashes', color='#4C72B0')
if not severe_sample.empty:
    plt.scatter(severe_sample['LONGITUDE'], severe_sample['LATITUDE'], s=6, alpha=0.45, label='Severe crashes', color='#C44E52')
plt.title('Crash Spatial Distribution (Sample)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(loc='best')
save_fig('05_spatial_scatter_sample.png')
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
plt.hexbin(spatial['LONGITUDE'], spatial['LATITUDE'], gridsize=80, cmap='viridis', mincnt=1)
plt.colorbar(label='Crash count per hex')
plt.title('Hexbin Crash Density')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
save_fig('06_hexbin_crash_density.png')
plt.show()

severe_spatial = spatial[spatial['is_severe'] == 1]
if len(severe_spatial) > 0:
    plt.figure(figsize=(10, 8))
    plt.hexbin(severe_spatial['LONGITUDE'], severe_spatial['LATITUDE'], gridsize=80, cmap='magma', mincnt=1)
    plt.colorbar(label='Severe crash count per hex')
    plt.title('Hexbin Severe Crash Density')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    save_fig('07_hexbin_severe_density.png')
    plt.show()


In [ ]:
spatial = spatial.copy()

# 1. Create smaller spatial grid
GRID_SIZE = 0.0025   # smaller than 0.01, roughly ~500m level because latitude: 0.01 ≈ 1.1 km
#longitude in Chicago: 0.01 ≈ 0.8 kmlongitude in Chicago: 0.01 ≈ 0.8 km

spatial["lat_bin"] = (spatial["LATITUDE"] / GRID_SIZE).round() * GRID_SIZE
spatial["lon_bin"] = (spatial["LONGITUDE"] / GRID_SIZE).round() * GRID_SIZE

# 2. Aggregate becauuse severe_rate_pct can over-rank low-volume cells, misleading
grid = (
    spatial.groupby(["lat_bin", "lon_bin"], as_index=False)
    .agg(
        crash_count=("is_severe", "size"),
        severe_count=("is_severe", "sum"),
        severe_rate=("is_severe", "mean")
    ))

grid["severe_rate_pct"] = grid["severe_rate"] * 100

# 3. Keep stable cells
grid = grid[grid["crash_count"] >= 20].copy()

# 4. Shrink severe rate for stability
#    (helps avoid over-ranking small cells)
global_rate = spatial["is_severe"].mean()
alpha = 30

grid["severe_rate_shrunk"] = (grid["severe_count"] + alpha * global_rate) / (grid["crash_count"] + alpha)
grid["severe_rate_shrunk_pct"] = grid["severe_rate_shrunk"] * 100

# 5. Rank hotspots
hotspots = grid.sort_values(["severe_rate_shrunk_pct", "crash_count"],
    ascending=[False, False]).head(12)

print("Top stable grid hotspots (>=20 crashes):")
print(hotspots[
        ["lat_bin", "lon_bin", "crash_count", "severe_count",
         "severe_rate_pct", "severe_rate_shrunk_pct"]
    ].to_string(index=False))

# 6. Visualize
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    grid["lon_bin"],
    grid["lat_bin"],
    c=grid["severe_rate_shrunk_pct"],
    s=grid["crash_count"],
    cmap="inferno",
    alpha=0.75
)

plt.colorbar(scatter, label="Shrunk Severe Rate (%)")
plt.title("Spatial Severe Crash Risk by Grid Cell")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

severe_count--> tells whether a hotspot has real harm volume.
severe_rate_shrunk --> reduces instability from medium-small cells.
smaller GRID_SIZE --> This gets closer to the scale of actual interventions.
map --> to see whether hotspots cluster spatially.

Crash locations were aggregated into geographic grid cells to identify stable spatial hotspots. For each cell, we computed total crash count, severe crash count, and severe crash rate. To reduce the influence of small-sample volatility, we also estimated a shrunk severe rate that pulls low-exposure cells toward the citywide average. Cells with at least 30 crashes were treated as stable candidates for hotspot review.

## 7) Aggregation preview for condition-level risk modeling

This section makes the condition-profile view easier to interpret by separating three questions:
1. How does severe rate behave across exposure levels?
2. Which weather-lighting combinations carry higher weighted risk?
3. Which condition profiles stay high-risk after applying a stability adjustment?


In [ ]:
#7) Aggregation preview for condition-level risk modeling
# Goal: summarize into location-level and condition-level profiles that preview
# which combinations of risk factors are associated with higher severe crash rates before formal modeling.

preview = train_fe.copy()

# Spatial grid
# Use a moderate grid size for interpretable hotspot previews.
GRID_SIZE = 0.005   # to approximate intersection-level clusters , nearby cells (~500 m).

preview['grid_lat'] = (pd.to_numeric(preview['LATITUDE'], errors='coerce') / GRID_SIZE).round() * GRID_SIZE
preview['grid_lon'] = (pd.to_numeric(preview['LONGITUDE'], errors='coerce') / GRID_SIZE).round() * GRID_SIZE

if 'weather_bucket' not in preview.columns:
    weather_src = preview.get('WEATHER_CONDITION', pd.Series('UNKNOWN', index=preview.index)).astype(str).str.upper()

    preview['weather_bucket'] = np.select(
        [
            weather_src.str.contains('CLEAR'),
            weather_src.str.contains('CLOUD'),
            weather_src.str.contains('RAIN|DRIZZLE'),
            weather_src.str.contains('SNOW|SLEET|ICE|FREEZING|BLOWING SNOW'),
            weather_src.str.contains('FOG|SMOKE|HAZE'),
            weather_src.eq('UNKNOWN')
        ],
        ['CLEAR', 'CLOUDY', 'RAIN', 'SNOW_ICE', 'LOW_VISIBILITY', 'UNKNOWN'],
        default='OTHER'
    )

# 7.2 Create lighting bucket consistently
if 'lighting_bucket' not in preview.columns:
    lighting_src = preview.get('LIGHTING_CONDITION', pd.Series('UNKNOWN', index=preview.index)).astype(str).str.upper()

    preview['lighting_bucket'] = np.select(
        [
            lighting_src.str.contains('DAYLIGHT'),
            lighting_src.str.contains('DARKNESS, LIGHTED'),
            lighting_src.str.contains('DARKNESS') & ~lighting_src.str.contains('LIGHTED'),
            lighting_src.str.contains('DAWN|DUSK'),
            lighting_src.eq('UNKNOWN')
        ],
        ['DAYLIGHT', 'DARK_LIT', 'DARK_UNLIT', 'DAWN_DUSK', 'UNKNOWN'],
        default='OTHER'
    )

if 'time_bucket' not in preview.columns:
    preview['time_bucket'] = np.select(
        [
            preview.get('is_weekend', 0).astype(int) == 1,
            preview.get('is_rush_hour', 0).astype(int) == 1,
            preview.get('is_night', 0).astype(int) == 1,
        ],
        ['WEEKEND', 'RUSH_HOUR', 'NIGHT'],
        default='DAYTIME'
    )

if 'speed_context' not in preview.columns:
    speed_src = pd.to_numeric(preview.get('POSTED_SPEED_LIMIT', np.nan), errors='coerce')

    preview['speed_context'] = np.select(
        [
            speed_src >= 40,
            speed_src >= 30,
            speed_src.notna()
        ],
        ['HIGH_SPEED', 'MID_SPEED', 'LOW_SPEED'],
        default='UNKNOWN'
    )

# Aggregation

agg_preview = (
    preview.dropna(subset=['grid_lat', 'grid_lon'])
           .groupby(
               [
                   'grid_lat', 'grid_lon',
                   'weather_bucket', 'lighting_bucket',
                   'time_bucket', 'speed_context'
               ],
               as_index=False
           )
           .agg(
               crash_count=('is_severe', 'size'),
               severe_count=('is_severe', 'sum'),
               severe_rate=('is_severe', 'mean'),
               avg_posted_speed=('POSTED_SPEED_LIMIT', 'mean'),
               pct_intersection=('is_intersection', 'mean') if 'is_intersection' in preview.columns else ('is_severe', 'mean'),
               pct_hit_run=('is_hit_and_run', 'mean') if 'is_hit_and_run' in preview.columns else ('is_severe', 'mean'),
               pct_self_reported=('is_self_reported', 'mean') if 'is_self_reported' in preview.columns else ('is_severe', 'mean'),
               pct_night=('is_night', 'mean') if 'is_night' in preview.columns else ('is_severe', 'mean'),
               pct_weekend=('is_weekend', 'mean') if 'is_weekend' in preview.columns else ('is_severe', 'mean'),
               pct_rush_hour=('is_rush_hour', 'mean') if 'is_rush_hour' in preview.columns else ('is_severe', 'mean'),
           )
)

agg_preview['severe_rate_pct'] = agg_preview['severe_rate'] * 100

# Empirical-Bayes shrinkage for stable ranking
global_rate = preview['is_severe'].mean()
alpha = 30

agg_preview['severe_rate_shrunk'] = (agg_preview['severe_count'] + alpha * global_rate) /(agg_preview['crash_count'] + alpha)
agg_preview['severe_rate_shrunk_pct'] = agg_preview['severe_rate_shrunk'] * 100

# Stable profiles
stable_profiles = agg_preview[agg_preview['crash_count'] >= 30].copy()

print('Total condition profiles:', len(agg_preview))
print('Stable profiles (crash_count >= 30):', len(stable_profiles))
print('Global severe rate (%):', round(global_rate * 100, 3))
print('Crash-count quantiles (stable profiles):')
print(stable_profiles['crash_count'].quantile([0.25, 0.5, 0.75, 0.9, 0.95]).to_string())

Empirical-Bayes shrinkage was applied to stabilize severe-rate estimates for condition profiles with limited exposure. <br>
The shrinkage parameter α controls the strength of the global prior. We set α = 30, which corresponds to approximately 30 pseudo-observations drawn from the citywide severe crash rate. This value aligns with the exposure stability diagnostic showing that severe-rate estimates become reasonably stable once crash counts exceed roughly 20–30 observations. <br>
Using α = 30 therefore balances variance reduction for sparse profiles while preserving meaningful differences among higher-exposure conditions.<br>

Total condition profiles ; 142920 means the aggregation created 142,920 different condition environments, which comes from 

grid_lat
grid_lon
weather_bucket
lighting_bucket
time_bucket
speed_context

So your 

In [ ]:
# Exposure stability diagnostic
plt.figure(figsize=(10,6))

plt.scatter(
    agg_preview['crash_count'],
    agg_preview['severe_rate_pct'],
    alpha=0.3,
    s=15
)

plt.xscale('log')

plt.xlabel("Crash count per profile (log scale)")
plt.ylabel("Severe rate (%)")
plt.title("Severe Rate Stability vs Exposure")

# stability reference lines
plt.axvline(10, linestyle='--', color='orange', label='n = 10')
plt.axvline(20, linestyle='--', color='red', label='n = 20')
plt.axvline(30, linestyle='--', color='purple', label='n = 30')
plt.axvline(50, linestyle='--', color='green', label='n = 50')

plt.legend()

plt.show()

The stability diagnostic shows that severe crash rate estimates fluctuate substantially for profiles with fewer than 20 observations. 
Variability decreases significantly between 20 and 30 crashes, suggesting that profiles with at least 30 crashes provide more reliable estimates. 
For exploratory condition-level analysis, a minimum exposure threshold of 30 crashes was therefore considered appropriate, while a stricter threshold of 50 crashes was used when ranking the highest-risk profiles. <BR>
it justifies for using thereshold for stable_profiles >= 30 and for rank_profiles >= 50

In [ ]:
#Quick diagnostic that checks whether your stable profiles are dominated by only a few locations, which can silently bias clustering results:<BR>
#Are stable profiles concentrated in only a few locations?
# Count how many profiles come from each grid cell
cell_profile_counts = (
    stable_profiles
    .groupby(['grid_lat', 'grid_lon'])
    .size()
    .reset_index(name='n_profiles')
)

print("Number of grid cells represented:", len(cell_profile_counts))
print("Total stable profiles:", len(stable_profiles))

display(cell_profile_counts['n_profiles'].describe())

#Compute share of profiles from largest location
print("Share of profiles from largest cell: {:.2f}%".format(
    cell_profile_counts['n_profiles'].max() / cell_profile_counts['n_profiles'].sum() * 100
))

cell_profile_counts_sorted = cell_profile_counts.sort_values(
    'n_profiles',
    ascending=False
)

cell_profile_counts_sorted['cum_profiles'] = (
    cell_profile_counts_sorted['n_profiles'].cumsum()
)

cell_profile_counts_sorted['cum_share'] = (
    cell_profile_counts_sorted['cum_profiles'] /
    cell_profile_counts_sorted['n_profiles'].sum()
)

plt.figure(figsize=(6,6))

plt.plot(
    range(len(cell_profile_counts_sorted)),
    cell_profile_counts_sorted['cum_share']
)

plt.xlabel('Grid cells (sorted by number of profiles)')
plt.ylabel('Cumulative share of profiles')
plt.title('Spatial dominance of condition profiles')

plt.show()


0.05% shows that the top cell contributes a very small share of the profiles, indicating good spatial diversity and low risk of location bias in the profile<BR>
Stable crash condition profiles are broadly distributed across nearly 500 spatial grid cells, with each location contributing only a small number of profiles (median = 2, maximum = 5). <BR>
This indicates that the aggregated dataset is not dominated by a small number of geographic locations and therefore is suitable for condition-level clustering analysis.

In [ ]:
# 7.1 Exposure-binned severe-rate trend (weighted)

exp_bin_count = min(8, stable_profiles['crash_count'].nunique())
stable_profiles['exposure_bin'] = pd.qcut(stable_profiles['crash_count'],q=exp_bin_count,duplicates='drop')

exposure_bin_stats = (
    stable_profiles.groupby('exposure_bin', as_index=False, observed=False)
                   .agg(
                       n_profiles=('crash_count', 'size'),
                       median_exposure=('crash_count', 'median'),
                       severe_count=('severe_count', 'sum'),
                       crash_count=('crash_count', 'sum'),
                   )
)

exposure_bin_stats['weighted_severe_rate_pct'] = (100 * exposure_bin_stats['severe_count'] / exposure_bin_stats['crash_count'])

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(
    exposure_bin_stats['median_exposure'],
    exposure_bin_stats['weighted_severe_rate_pct'],
    marker='o',
    color='#C44E52'
)
ax1.set_xscale('log')
ax1.set_xlabel('Median crash count within exposure bin (log scale)')
ax1.set_ylabel('Weighted severe rate (%)', color='#C44E52')
ax1.tick_params(axis='y', labelcolor='#C44E52')
ax1.set_title('Exposure Bins vs Weighted Severe Rate')

ax2 = ax1.twinx()
ax2.bar(
    exposure_bin_stats['median_exposure'],
    exposure_bin_stats['n_profiles'],
    alpha=0.25,
    color='#4C72B0',
    width=6
)
ax2.set_ylabel('Number of profiles', color='#4C72B0')
ax2.tick_params(axis='y', labelcolor='#4C72B0')

save_fig('17_aggregation_exposure_bin_trend.png')
plt.show()

display(exposure_bin_stats)

weighted_severe_rate_pct --> tells whether severe risk persists as exposure rises helps distinguish structural vs sparse risk

This chart of Exposure Bins vs Weighted Severe Rate combines three pieces of information:
X-axis : Median crash count within exposure bin (log scale). This represents how many crashes occur in each condition profile group. So each bin contains profiles with similar crash exposure. Example: 

Blue bars:  Number of profiles, which shows how many condition profiles fall into each exposure bin. 
Example: exposure bin ~31–34 crashes has 157 profile. 
Profiles are not dominated by one exposure level

Red line: Weighted severe rate (%)
Weighted severe rate =∑severe crashes divided by ∑crashes, 

Key Question: Does severe crash risk increase with exposure?
Weighted severe rate were in range of ~1.2% – 1.9%, which indicates that the more number of crash doesnt contribute to severeness of the crash.
Instead, severity appears to depend more on environmental and roadway conditions, supporting the need for condition-level risk modeling rather than relying solely on crash frequency.


In [ ]:
# 7.2 Weather x lighting weighted severe-rate heatmap
wl = (
    stable_profiles.groupby(['weather_bucket', 'lighting_bucket'], as_index=False)
                   .agg(
                       severe_count=('severe_count', 'sum'),
                       crash_count=('crash_count', 'sum'),
                       n_profiles=('crash_count', 'size')
                   )
)

wl['weighted_severe_rate_pct'] = 100 * wl['severe_count'] / wl['crash_count']

wl_heat = wl.pivot(
    index='weather_bucket',
    columns='lighting_bucket',
    values='weighted_severe_rate_pct'
)

plt.figure(figsize=(9, 5))
sns.heatmap(
    wl_heat,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Weighted severe rate (%)'}
)
plt.title('Weather × Lighting Weighted Severe Rate (Stable Profiles)')
plt.xlabel('Lighting bucket')
plt.ylabel('Weather bucket')
save_fig('18_aggregation_weather_lighting_heatmap.png')
plt.show()

wl_rank = wl.sort_values(['weighted_severe_rate_pct', 'crash_count'],
    ascending=[False, False]
)

print('Top weather-lighting combinations (weighted severe rate):')
display(wl_rank.head(10))

This heatmap summarizes how severe crash risk changes across weather and lighting conditions using your stable condition profiles.
Interpretation of weighted_severe_rate_pct by weather × lighting: identifies risky environmental condition combinations.
This aligns with your earlier finding:

CLEAR | DARK_LIT = 2.30 was the highest-risk profiles. 
Meaning: Under clear weather at night with street lighting, about 2.30% of crashes are severe.Even with lighting, night conditions increase risk.
Possible explanation: nighttime visibility limitations; higher speeds; driver fatigue; or lower traffic enforcement

CLEAR | DAYLIGHT = 1.43%, meaning that during clear daytime conditions, severe crashes occur in about 1.43% of crashes.
This is lower than nighttime conditions, which is expected.
Explanation: Better visibility reduces severe crash risk.

RAIN | DARK_LIT = 0.00%, meaning that among the stable profiles matching this condition, no severe crashes occurred.
Meaning : This does not necessarily mean rain is safer.
Possible explanation: very small number of profiles, low crash counts in this condition, or drivers slow down during rain.

RAIN | DARK_LIT = "blank cell", which means that there were no weather × lighting profiles with ≥30 crashes for rain in daylight conditions.

Key takeawys:
This heatmap suggests that lighting conditions (day vs night) appear to influence crash severity more strongly than weather conditions in the stable profile dataset, with nighttime crashes showing higher severe crash rates than daytime crashes.

Important limitation of this figure : This heatmap is based on stable profiles only with crash_counts ≥30 crashes.
So, the heatmap reflects high-exposure environments, not all possible environments.

In [ ]:
# 7.3 Stable high-risk profile ranking
#using EBS method to create EBS priority ranking
rank_profiles = stable_profiles[stable_profiles['crash_count'] >= 50].copy()

# Create profile label for interpretability
rank_profiles['profile_label'] = (
    rank_profiles['weather_bucket'].astype(str) + ' | ' +
    rank_profiles['lighting_bucket'].astype(str) + ' | ' +
    rank_profiles['time_bucket'].astype(str) + ' | ' +
    rank_profiles['speed_context'].astype(str)
)

# Rank profiles using Empirical Bayes shrunk severe rate
rank_profiles = rank_profiles.sort_values(['severe_rate_shrunk_pct', 'crash_count'],ascending=[False, False]).reset_index(drop=True)

# Priority rank #each stable profile has its priority rank
rank_profiles['priority_rank'] = np.arange(1, len(rank_profiles) + 1)

# Convert rank to priority tier
def assign_priority_tier(rank):
    if rank <= 20:
        return 'CRITICAL'
    elif rank <= 100:
        return 'HIGH'
    elif rank <= 300:
        return 'MODERATE'
    else:
        return 'LOW'

rank_profiles['priority_tier'] = rank_profiles['priority_rank'].apply(assign_priority_tier)

# Show top profiles
top_profiles = rank_profiles.head(15).copy()

# Plotting dataframe
plot_df = top_profiles.sort_values('severe_rate_shrunk_pct', ascending=True)

plt.figure(figsize=(12, 8))

bars = plt.barh(
    plot_df['profile_label'],
    plot_df['severe_rate_shrunk_pct'],
    color='#4C72B0'
)

# Add labels inside bars
for bar, n, tier in zip(
    bars,
    plot_df['crash_count'],
    plot_df['priority_tier']
):
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    plt.text(
        width - 0.05,
        y,
        f'{width:.2f}% | n={int(n)} | {tier}',
        va='center',
        ha='right',
        color='white',
        fontsize=10
    )

plt.xlabel('Shrunk severe rate (%)')
plt.ylabel('Condition profile')
plt.title('Top Stable Condition Profiles by Shrunk Severe Rate')
save_fig('19_aggregation_top_condition_profiles.png')
plt.show()

# Save full priority ranking
priority_path = DATA_ROOT / 'cleaned_data' / 'aggregation_priority_profiles.csv'
rank_profiles.to_csv(priority_path, index=False)
print("Saved priority ranking:", priority_path.resolve())

# Save top profiles
top_path = DATA_ROOT / 'cleaned_data' / 'aggregation_preview_top_profiles.csv'
top_profiles.to_csv(top_path, index=False)
print('Saved top profiles:', top_path.resolve())

display(
    top_profiles[
        [
            'priority_rank',
            'priority_tier',
            'grid_lat', 'grid_lon',
            'weather_bucket', 'lighting_bucket', 'time_bucket', 'speed_context',
            'crash_count', 'severe_count',
            'severe_rate_pct', 'severe_rate_shrunk_pct'
        ]
    ]
)

The Top Stable Condition Profiles Ranking is based on Empirical Bayes Shrunk Severe Rate, which means severity risk adjusted for small sample bias.<BR>
Example: values of 5.61%, 4.37%, are are statistically stabilized severe crash probabilities.<BR>
The global rate earlier was ~1.65%, so these profiles are 2–3× higher than average.

The highest Profile: CLEAR | DARK_LIT | WEEKEND | MID_SPEED = 5.6%, which means Crashes under these conditions are much more likely to become severe<BR>
Possible possibility, such as nighttime driving, weekend recreational activity, higher speeds, alcohol or fatigue, lower traffic enforcement often produce high-energy collisions.


2nd highest profile : CLEAR | DAYLIGHT | RUSH_HOUR | MID_SPEED = 3.8%.
Meaning: Even though rush hour has lower speeds, the crash environment still produces elevated severity.
Possible causes: congestion; aggressive lane changes; rear-end chain crashes; intersection conflicts.

3rd highest profile : CLEAR | DAYLIGHT | DAYTIME | MID_SPEED = 3.2%
Meaning: This suggests the normal time conditions also still produces elevated severity.
Possible explanation: it could be road design, because Mid-speed urban arterials (30–40 mph) often have intersections, turning traffic, pedestrian crossings, or signal conflicts.<BR>

Key Findings:
MUrban mid-speed roads (30–40 mph) are a consistent severity environment despite whether it was rush hour or daytime. <BR>
Therefore, the environment matters more than the weather (weather is constant on CLEAR on those three highest profile)<BR>
This suggests roadway structure and traffic context drive severity.

Initial Hypothesis/ analysis suggests CDOT should prioritize interventions in: mid-speed arterial roads, then especially during nighttime or weekend periods.


## 8) Key takeaways for policy and modeling handoff


Summary notes:
1. Class imbalance remains strong and should be handled explicitly in modeling.
2. Temporal risk concentration appears in specific hours and day patterns, useful for deployment timing.
3. Categorical context (weather, lighting, crash type, traffic control, causes) shows meaningful risk variation.
4. Numerical signals (speed context, unit count, selected correlations) are informative but should be interpreted with effect size and exposure.
5. Spatial concentration persists in a subset of grid cells, supporting location-prioritized intervention.
6. Aggregated profile analysis improves interpretability by combining stability filters, weighted rates, and profile-level ranking.
7. EDA figures are exported to `image/02_eda/` for paper and presentation use.


## 3) Classification

**Owner:** Diyouva Novith

## Alignment Notes
- Core question: How well can we predict severe crash outcomes from crash-level features?
- Predicting tasks:
  - Track A: Predict `is_severe` from engineered crash-level features.
  - Track B: Predict condition-profile risk from aggregated profiles.
- Based on what data:
  - `dataset/cleaned_data/train_model.csv`, `test_model.csv` (from notebook 02)
  - `dataset/cleaned_data/train.csv`, `test.csv` (from notebook 01)
  - `dataset/cleaned_data/feature_columns.json` (feature contract from notebook 02)
- Outputs:
  - `dataset/cleaned_data/risk_scores.csv` for notebooks 04 and 05
  - Model evaluation tables and figures for paper
- Run-order dependency: `01 -> 02 -> 03 -> 04 -> 05`

---

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
CLEAN_DIR = DATA_ROOT / 'cleaned_data'
IMAGE_DIR = MAIN_IMAGE_DIR
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(filename: str, dpi: int = 180) -> Path:
    out = IMAGE_DIR / filename
    plt.savefig(out, dpi=dpi, bbox_inches='tight')
    print('Saved figure:', out.resolve())
    return out


try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

## 1) Data continuity contract (JSON + CSV)

In [ ]:
required_inputs = {
    'train_model.csv': CLEAN_DIR / 'train_model.csv',
    'test_model.csv': CLEAN_DIR / 'test_model.csv',
    'train.csv': CLEAN_DIR / 'train.csv',
    'test.csv': CLEAN_DIR / 'test.csv',
    'feature_columns.json': CLEAN_DIR / 'feature_columns.json',
}

for name, path in required_inputs.items():
    status = 'exists' if path.exists() else 'MISSING'
    print(f'  {name}: {status}')
    assert path.exists(), f'Required input missing: {path}'

# Load feature contract
with open(CLEAN_DIR / 'feature_columns.json') as f:
    meta = json.load(f)

FEATURE_COLS = meta['feature_cols']
TARGET = meta['target']
print(f'\nFeature contract: {len(FEATURE_COLS)} features, target={TARGET}')

In [ ]:
# Load engineered model data
train_model = pd.read_csv(CLEAN_DIR / 'train_model.csv')
test_model = pd.read_csv(CLEAN_DIR / 'test_model.csv')

# Enforce contract: feature_cols must be present
missing_train = set(FEATURE_COLS) - set(train_model.columns)
missing_test = set(FEATURE_COLS) - set(test_model.columns)
assert not missing_train, f'Missing in train_model: {missing_train}'
assert not missing_test, f'Missing in test_model: {missing_test}'

X_train = train_model[FEATURE_COLS]
y_train = train_model[TARGET]
X_test = test_model[FEATURE_COLS]
y_test = test_model[TARGET]

print(f'Train: {X_train.shape}, severe rate: {y_train.mean()*100:.2f}% ({y_train.sum()} / {len(y_train)})')
print(f'Test:  {X_test.shape}, severe rate: {y_test.mean()*100:.2f}% ({y_test.sum()} / {len(y_test)})')

---
## 2) Track A — Crash-Level Classification

### 2.1 Handle class imbalance

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE oversampling (30% minority ratio) for LR
smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
print(f'After SMOTE:  {dict(pd.Series(y_train_smote).value_counts())}')

# scale_pos_weight for XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'\nscale_pos_weight for XGBoost: {scale_pos:.1f}')

### 2.2 Model training

In [ ]:
results = {}

In [ ]:
%%time
# --- Logistic Regression (baseline) ---
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, solver='lbfgs')
lr.fit(X_train_smote, y_train_smote)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

results['Logistic Regression'] = {'model': lr, 'y_pred': y_pred_lr, 'y_prob': y_prob_lr}

print('Logistic Regression (SMOTE + balanced)')
print(classification_report(y_test, y_pred_lr, target_names=['Non-Severe', 'Severe']))

In [ ]:
%%time
# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=15,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

results['Random Forest'] = {'model': rf, 'y_pred': y_pred_rf, 'y_prob': y_prob_rf}

print('Random Forest (balanced class weights)')
print(classification_report(y_test, y_pred_rf, target_names=['Non-Severe', 'Severe']))

In [ ]:
%%time
# --- XGBoost ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

results['XGBoost'] = {'model': xgb, 'y_pred': y_pred_xgb, 'y_prob': y_prob_xgb}

print('XGBoost (scale_pos_weight)')
print(classification_report(y_test, y_pred_xgb, target_names=['Non-Severe', 'Severe']))

### 2.3 Model evaluation

In [ ]:
# --- ROC and Precision-Recall Curves ---
colors = {'Logistic Regression': '#4C72B0', 'Random Forest': '#55A868', 'XGBoost': '#C44E52'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = roc_auc_score(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=colors[name], linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

for name, res in results.items():
    precision, recall, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    axes[1].plot(recall, precision, label=f'{name} (AP={ap:.4f})', color=colors[name], linewidth=2)

axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_fig('19_roc_pr_curves.png')
plt.show()

In [ ]:
# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Non-Severe', 'Severe'],
                yticklabels=['Non-Severe', 'Severe'])
    ax.set_title(name)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (Default Threshold)', y=1.02)
plt.tight_layout()
save_fig('20_confusion_matrices.png')
plt.show()

In [ ]:
# --- Threshold Analysis ---
best_model_name = max(results.keys(), key=lambda k: roc_auc_score(y_test, results[k]['y_prob']))
best_probs = results[best_model_name]['y_prob']

thresholds = np.arange(0.01, 0.50, 0.01)
f1_scores, recall_scores, precision_scores = [], [], []

for t in thresholds:
    y_pred_t = (best_probs >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    f1_scores.append(f1)
    recall_scores.append(rec)
    precision_scores.append(prec)

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, f1_scores, label='F1 (Severe)', linewidth=2, color='#C44E52')
ax.plot(thresholds, recall_scores, label='Recall (Severe)', linewidth=2, color='#55A868')
ax.plot(thresholds, precision_scores, label='Precision (Severe)', linewidth=2, color='#4C72B0')
ax.axvline(x=optimal_threshold, color='gray', linestyle='--', alpha=0.7,
           label=f'Optimal threshold={optimal_threshold:.2f}')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Threshold Analysis ({best_model_name})')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig('21_threshold_analysis.png')
plt.show()

print(f'\nOptimal threshold for {best_model_name}: {optimal_threshold:.2f}')
print(f'  F1:        {f1_scores[optimal_idx]:.4f}')
print(f'  Recall:    {recall_scores[optimal_idx]:.4f}')
print(f'  Precision: {precision_scores[optimal_idx]:.4f}')

In [ ]:
# --- Evaluation with Optimal Threshold ---
y_pred_optimal = (best_probs >= optimal_threshold).astype(int)

print(f'{best_model_name} with optimal threshold ({optimal_threshold:.2f}):')
print(classification_report(y_test, y_pred_optimal, target_names=['Non-Severe', 'Severe']))

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds', ax=ax,
            xticklabels=['Non-Severe', 'Severe'],
            yticklabels=['Non-Severe', 'Severe'])
ax.set_title(f'{best_model_name} (threshold={optimal_threshold:.2f})')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

plt.tight_layout()
save_fig('22_cm_optimal_threshold.png')
plt.show()

### 2.4 Feature importance

In [ ]:
# --- Random Forest Feature Importance ---
top_n = 25

rf_imp = pd.DataFrame({'feature': FEATURE_COLS, 'importance': rf.feature_importances_})
rf_imp = rf_imp.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(rf_imp.tail(top_n)['feature'], rf_imp.tail(top_n)['importance'], color='#55A868')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title(f'Random Forest: Top {top_n} Features')
plt.tight_layout()
save_fig('23_rf_feature_importance.png')
plt.show()

In [ ]:
# --- XGBoost Feature Importance ---
xgb_imp = pd.DataFrame({'feature': FEATURE_COLS, 'importance': xgb.feature_importances_})
xgb_imp = xgb_imp.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(xgb_imp.tail(top_n)['feature'], xgb_imp.tail(top_n)['importance'], color='#C44E52')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title(f'XGBoost: Top {top_n} Features')
plt.tight_layout()
save_fig('24_xgb_feature_importance.png')
plt.show()

In [ ]:
# --- Logistic Regression Coefficients ---
lr_coefs = pd.DataFrame({'feature': FEATURE_COLS, 'coefficient': lr.coef_[0]})
lr_coefs = lr_coefs.sort_values('coefficient')

top_pos = lr_coefs.tail(15)
top_neg = lr_coefs.head(15)
lr_display = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(10, 10))
colors_lr = ['#C44E52' if c > 0 else '#4C72B0' for c in lr_display['coefficient']]
ax.barh(lr_display['feature'], lr_display['coefficient'], color=colors_lr)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Coefficient (positive = increases severity risk)')
ax.set_title('Logistic Regression: Top Coefficients')
plt.tight_layout()
save_fig('25_lr_coefficients.png')
plt.show()

In [ ]:
# --- Cross-model feature importance overlap ---
rf_top15 = set(rf_imp.tail(15)['feature'])
xgb_top15 = set(xgb_imp.tail(15)['feature'])
common = rf_top15 & xgb_top15

print(f'Top-15 feature overlap between RF and XGBoost:')
print(f'  Common ({len(common)}): {sorted(common)}')
print(f'  RF only ({len(rf_top15 - xgb_top15)}): {sorted(rf_top15 - xgb_top15)}')
print(f'  XGBoost only ({len(xgb_top15 - rf_top15)}): {sorted(xgb_top15 - rf_top15)}')

### 2.5 Track A Summary

In [ ]:
summary_rows = []
for name, res in results.items():
    report = classification_report(y_test, res['y_pred'], labels=[0, 1], target_names=['Non-Severe', 'Severe'], zero_division=0, output_dict=True)
    auc = roc_auc_score(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    summary_rows.append({
        'Model': name,
        'Precision (Severe)': f"{report['Severe']['precision']:.4f}",
        'Recall (Severe)': f"{report['Severe']['recall']:.4f}",
        'F1 (Severe)': f"{report['Severe']['f1-score']:.4f}",
        'AUC-ROC': f'{auc:.4f}',
        'Avg Precision': f'{ap:.4f}'
    })

summary_df = pd.DataFrame(summary_rows)
print('=' * 90)
print('TRACK A: CRASH-LEVEL MODEL COMPARISON')
print('=' * 90)
display(summary_df)
print('=' * 90)
print(f'\nBest model by AUC-ROC: {best_model_name}')
print(f'Optimal threshold: {optimal_threshold:.2f}')
print(f'At optimal threshold - Recall: {recall_scores[optimal_idx]:.4f}, Precision: {precision_scores[optimal_idx]:.4f}, F1: {f1_scores[optimal_idx]:.4f}')

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(12, 6))

models = list(results.keys())
metrics_names = ['AUC-ROC', 'Recall (Severe)', 'F1 (Severe)']

metric_values = {}
for name, res in results.items():
    report = classification_report(y_test, res['y_pred'], labels=[0, 1], target_names=['Non-Severe', 'Severe'], zero_division=0, output_dict=True)
    metric_values[name] = [
        roc_auc_score(y_test, res['y_prob']),
        report['Severe']['recall'],
        report['Severe']['f1-score']
    ]

x = np.arange(len(metrics_names))
width = 0.25

for i, (name, vals) in enumerate(metric_values.items()):
    bars = ax.bar(x + i * width, vals, width, label=name,
                  color=list(colors.values())[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_names)
ax.set_ylabel('Score')
ax.set_title('Track A: Model Comparison')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig('26_model_comparison.png')
plt.show()

---
## 3) Track B — Condition-Level Risk Model

Aggregate crash data to condition profiles and rank risk for CDOT prioritization.

In [ ]:
# Load base train/test for aggregation
train_base = pd.read_csv(CLEAN_DIR / 'train.csv')
test_base = pd.read_csv(CLEAN_DIR / 'test.csv')
train_base['CRASH_DATE'] = pd.to_datetime(train_base['CRASH_DATE'], errors='coerce')
test_base['CRASH_DATE'] = pd.to_datetime(test_base['CRASH_DATE'], errors='coerce')

# Combine for profile building (use full data for stable profiles)
all_base = pd.concat([train_base, test_base], ignore_index=True)

print(f'Combined base data: {len(all_base)} rows')

In [ ]:
# Build condition profiles
profile = all_base.copy()
profile = profile.dropna(subset=['LATITUDE', 'LONGITUDE'])

# Grid cells (~500m)
profile['grid_lat'] = (pd.to_numeric(profile['LATITUDE'], errors='coerce') / 0.005).round() * 0.005
profile['grid_lon'] = (pd.to_numeric(profile['LONGITUDE'], errors='coerce') / 0.005).round() * 0.005

# Weather bucket
weather_u = profile['WEATHER_CONDITION'].fillna('UNKNOWN').astype(str).str.upper()
profile['weather_bucket'] = np.select(
    [weather_u.str.contains('CLEAR'), weather_u.str.contains('RAIN|DRIZZLE'),
     weather_u.str.contains('SNOW|SLEET|ICE|FREEZING')],
    ['CLEAR', 'RAIN', 'SNOW_ICE'], default='OTHER')

# Lighting bucket
lighting_u = profile['LIGHTING_CONDITION'].fillna('UNKNOWN').astype(str).str.upper()
profile['lighting_bucket'] = np.select(
    [lighting_u.str.contains('DAYLIGHT'), lighting_u.str.contains('DARKNESS, LIGHTED'),
     lighting_u.str.contains('DARKNESS') & ~lighting_u.str.contains('LIGHTED'),
     lighting_u.str.contains('DAWN|DUSK')],
    ['DAYLIGHT', 'DARK_LIT', 'DARK_UNLIT', 'DAWN_DUSK'], default='OTHER')

# Time bucket
hour = pd.to_numeric(profile['CRASH_HOUR'], errors='coerce').fillna(12)
dow = pd.to_numeric(profile['CRASH_DAY_OF_WEEK'], errors='coerce').fillna(2)
profile['time_bucket'] = np.select(
    [dow.isin([1, 7]), hour.isin([7,8,9,16,17,18]),
     hour.isin([20,21,22,23,0,1,2,3,4,5])],
    ['WEEKEND', 'RUSH_HOUR', 'NIGHT'], default='DAYTIME')

# Speed bucket
speed = pd.to_numeric(profile['POSTED_SPEED_LIMIT'], errors='coerce').fillna(30)
profile['speed_bucket'] = pd.cut(speed, bins=[-1, 25, 30, 35, 45, 100],
                                  labels=['<=25', '26-30', '31-35', '36-45', '46+']).astype(str)

# Aggregate
agg_cols = ['grid_lat', 'grid_lon', 'weather_bucket', 'lighting_bucket', 'time_bucket', 'speed_bucket']
agg = (profile.groupby(agg_cols, as_index=False)
       .agg(crash_count=('is_severe', 'size'),
            severe_count=('is_severe', 'sum'),
            severe_rate=('is_severe', 'mean')))

print(f'Total condition profiles: {len(agg)}')
print(f'Profiles with >=10 crashes: {(agg["crash_count"] >= 10).sum()}')

In [ ]:
# Empirical-Bayes shrinkage for stable ranking
global_rate = profile['is_severe'].mean()
alpha = 30  # shrinkage strength

agg['predicted_severe_rate'] = (agg['severe_count'] + alpha * global_rate) / (agg['crash_count'] + alpha)
agg['has_severe'] = (agg['severe_count'] > 0).astype(int)

# Risk score = shrunk rate * log(crash_count + 1) to weight both rate and volume
agg['risk_score'] = agg['predicted_severe_rate'] * np.log1p(agg['crash_count'])

# Rank
agg = agg.sort_values('risk_score', ascending=False).reset_index(drop=True)
agg['rank'] = range(1, len(agg) + 1)

# Recommendation
def assign_recommendation(row):
    if row['rank'] <= 50:
        return 'PRIORITY: Immediate infrastructure review + targeted enforcement'
    elif row['rank'] <= 200:
        return 'HIGH: Schedule safety audit and consider countermeasures'
    elif row['rank'] <= 500:
        return 'MODERATE: Monitor and include in next planning cycle'
    else:
        return 'STANDARD: Routine monitoring'

agg['recommendation'] = agg.apply(assign_recommendation, axis=1)

print('Risk score distribution:')
display(agg[['risk_score', 'crash_count', 'severe_rate', 'predicted_severe_rate', 'rank']].describe())

print(f'\nTop 10 highest-risk profiles:')
display(agg.head(10)[['grid_lat', 'grid_lon', 'weather_bucket', 'lighting_bucket',
                       'time_bucket', 'speed_bucket', 'crash_count', 'severe_count',
                       'severe_rate', 'predicted_severe_rate', 'risk_score', 'rank', 'recommendation']])

In [ ]:
# Visualize top risk profiles
top_profiles = agg.head(20).copy()
top_profiles['label'] = (
    top_profiles['weather_bucket'] + ' | ' +
    top_profiles['lighting_bucket'] + ' | ' +
    top_profiles['time_bucket'] + ' | ' +
    top_profiles['grid_lat'].round(3).astype(str) + ',' +
    top_profiles['grid_lon'].round(3).astype(str)
)

plot_df = top_profiles.sort_values('risk_score', ascending=True)
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(plot_df['label'], plot_df['risk_score'], color='#C44E52')
for bar, n in zip(bars, plot_df['crash_count']):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' n={int(n)}', va='center', fontsize=9)
ax.set_xlabel('Risk Score')
ax.set_title('Track B: Top 20 Condition Profiles by Risk Score')
plt.tight_layout()
save_fig('27_top_risk_profiles.png')
plt.show()

In [ ]:
# Save risk_scores.csv for notebooks 04 and 05
risk_output_cols = [
    'grid_lat', 'grid_lon',
    'weather_bucket', 'lighting_bucket', 'time_bucket', 'speed_bucket',
    'crash_count', 'severe_count', 'severe_rate',
    'predicted_severe_rate', 'risk_score', 'rank', 'recommendation'
]

risk_scores_path = CLEAN_DIR / 'risk_scores.csv'
agg[risk_output_cols].to_csv(risk_scores_path, index=False)

print(f'Saved: {risk_scores_path.resolve()}')
print(f'  Rows: {len(agg)}')
print(f'  Size: {risk_scores_path.stat().st_size / 1024:.1f} KB')
print(f'  Columns: {risk_output_cols}')

---
## 4) Handoff summary

In [ ]:
print('=' * 70)
print('NOTEBOOK 03 OUTPUTS')
print('=' * 70)
print(f'''
Track A (Crash-Level Classification):
  Best model: {best_model_name}
  AUC-ROC: {roc_auc_score(y_test, results[best_model_name]["y_prob"]):.4f}
  Optimal threshold: {optimal_threshold:.2f}
  At optimal threshold:
    Recall:    {recall_scores[optimal_idx]:.4f}
    Precision: {precision_scores[optimal_idx]:.4f}
    F1:        {f1_scores[optimal_idx]:.4f}

Track B (Condition-Level Risk):
  Total profiles: {len(agg)}
  Priority profiles (rank<=50): {(agg["rank"] <= 50).sum()}
  High-risk profiles (rank<=200): {(agg["rank"] <= 200).sum()}
  Output: risk_scores.csv

Figures saved to: {IMAGE_DIR.resolve()}

Handoff to downstream notebooks:
  - 04_clustering.ipynb: reads train.csv for spatial clustering
  - 05_spatial_dashboard.ipynb: reads train.csv + risk_scores.csv + cluster_labels.csv
''')

## 4) Spatial Clustering

## Alignment Notes
- Core question: Do grid cells form distinct risk clusters and which clusters should be prioritized?
- Based on what:
  - `dataset/cleaned_data/train.csv` (base crash data from notebook 01)
  - `dataset/cleaned_data/risk_scores.csv` (risk profile output from notebook 03)
- Unit of analysis: grid cell (must be consistent with notebook 03 profile grid).
- Output expected:
  - `dataset/cleaned_data/cluster_labels.csv` for notebook 05
- Run-order dependency: notebook 03 must complete and produce `risk_scores.csv` first.

In [ ]:
# Placeholder-only notebook: implementation deferred.
from pathlib import Path
import pandas as pd
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA


## 1) Input continuity checks (TODO)


In [ ]:
DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
CLEAN_DIR = DATA_ROOT / 'cleaned_data'

train_path = CLEAN_DIR / 'train.csv'
risk_path = CLEAN_DIR / 'risk_scores.csv'

print('train.csv', 'exists' if train_path.exists() else 'missing')
print('risk_scores.csv', 'exists' if risk_path.exists() else 'missing')

# TODO: when implementing, enforce required risk_scores columns:
required_risk_cols = [
    'grid_lat', 'grid_lon', 'risk_score', 'predicted_severe_rate',
    'crash_count', 'severe_count', 'rank'
]
print('required risk_scores columns:', required_risk_cols)


In [ ]:
# — Empirical Bayes Condition Risk Estimation

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
CLEAN_DIR = DATA_ROOT / 'cleaned_data'
IMAGE_DIR = MAIN_IMAGE_DIR
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name: str, dpi: int = 160) -> Path:
    out = IMAGE_DIR / name
    plt.savefig(out, dpi=dpi, bbox_inches='tight')
    print('Saved figure:', out.resolve())
    return out

# ------------------------------------------------------------
# Load base train/test
# ------------------------------------------------------------
train_base = pd.read_csv(CLEAN_DIR / 'train.csv')
test_base = pd.read_csv(CLEAN_DIR / 'test.csv')

for df in [train_base, test_base]:
    df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'], errors='coerce')

all_base = pd.concat([train_base, test_base], ignore_index=True)

print(f'Combined base rows: {len(all_base):,}')
print(f'Image output folder: {IMAGE_DIR.resolve()}')

# ------------------------------------------------------------
# Build profile base
# ------------------------------------------------------------
profile = all_base.copy()
profile = profile.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()

# IMPORTANT: use one grid size consistently across notebooks
GRID_SIZE = 0.005

profile['grid_lat'] = (pd.to_numeric(profile['LATITUDE'], errors='coerce') / GRID_SIZE).round() * GRID_SIZE
profile['grid_lon'] = (pd.to_numeric(profile['LONGITUDE'], errors='coerce') / GRID_SIZE).round() * GRID_SIZE

# ------------------------------------------------------------
# Feature engineering consistent with EDA

def normalize_text(series):
    return (
        series.fillna('UNKNOWN')
              .astype(str)
              .str.upper()
              .str.strip()
              .replace({'': 'UNKNOWN'})
    )

# Weather bucket
weather_u = normalize_text(profile['WEATHER_CONDITION']) if 'WEATHER_CONDITION' in profile.columns else pd.Series('UNKNOWN', index=profile.index)
profile['weather_bucket'] = np.select(
    [
        weather_u.str.contains('CLEAR'),
        weather_u.str.contains('CLOUD'),
        weather_u.str.contains('RAIN|DRIZZLE'),
        weather_u.str.contains('SNOW|SLEET|ICE|FREEZING|BLOWING SNOW'),
        weather_u.str.contains('FOG|SMOKE|HAZE'),
        weather_u.eq('UNKNOWN')
    ],
    ['CLEAR', 'CLOUDY', 'RAIN', 'SNOW_ICE', 'LOW_VISIBILITY', 'UNKNOWN'],
    default='OTHER'
)

# Lighting bucket
lighting_u = normalize_text(profile['LIGHTING_CONDITION']) if 'LIGHTING_CONDITION' in profile.columns else pd.Series('UNKNOWN', index=profile.index)
profile['lighting_bucket'] = np.select(
    [
        lighting_u.str.contains('DAYLIGHT'),
        lighting_u.str.contains('DARKNESS, LIGHTED'),
        lighting_u.str.contains('DARKNESS') & ~lighting_u.str.contains('LIGHTED'),
        lighting_u.str.contains('DAWN|DUSK'),
        lighting_u.eq('UNKNOWN')
    ],
    ['DAYLIGHT', 'DARK_LIT', 'DARK_UNLIT', 'DAWN_DUSK', 'UNKNOWN'],
    default='OTHER'
)

# Time bucket
hour = pd.to_numeric(profile['CRASH_HOUR'], errors='coerce').fillna(12)
dow = pd.to_numeric(profile['CRASH_DAY_OF_WEEK'], errors='coerce').fillna(2)

profile['time_bucket'] = np.select(
    [
        dow.isin([1, 7]),
        hour.isin([7, 8, 9, 16, 17, 18]),
        hour.isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5]),
    ],
    ['WEEKEND', 'RUSH_HOUR', 'NIGHT'],
    default='DAYTIME'
)

# Speed context
speed_src = pd.to_numeric(profile['POSTED_SPEED_LIMIT'], errors='coerce')
profile['speed_context'] = np.select(
    [
        speed_src >= 40,
        speed_src >= 30,
        speed_src.notna()
    ],
    ['HIGH_SPEED', 'MID_SPEED', 'LOW_SPEED'],
    default='UNKNOWN'
)

# Binary structure / behavior indicators
if 'INTERSECTION_RELATED_I' in profile.columns:
    profile['is_intersection'] = (
        profile['INTERSECTION_RELATED_I']
        .fillna('N')
        .astype(str)
        .str.upper()
        .str.strip()
        .eq('Y')
        .astype(int)
    )
else:
    profile['is_intersection'] = 0

if 'HIT_AND_RUN_I' in profile.columns:
    profile['is_hit_and_run'] = (
        profile['HIT_AND_RUN_I']
        .fillna('N')
        .astype(str)
        .str.upper()
        .str.strip()
        .eq('Y')
        .astype(int)
    )
else:
    profile['is_hit_and_run'] = 0

if 'REPORT_TYPE' in profile.columns:
    profile['is_self_reported'] = (
        profile['REPORT_TYPE']
        .astype(str)
        .str.upper()
        .str.contains('NOT ON SCENE', na=False)
        .astype(int)
    )
else:
    profile['is_self_reported'] = 0

profile['is_night'] = hour.isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
profile['is_rush_hour'] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)
profile['is_weekend'] = dow.isin([1, 7]).astype(int)

# Optional roadway context
if 'TRAFFICWAY_TYPE' in profile.columns:
    t = normalize_text(profile['TRAFFICWAY_TYPE'])
    profile['is_four_way'] = (t == 'FOUR WAY').astype(int)
    profile['is_five_point'] = t.str.contains('FIVE POINT', na=False).astype(int)
    profile['is_y_intersection'] = t.str.contains('Y-INTERSECTION', na=False).astype(int)
    profile['is_t_intersection'] = t.str.contains('T-INTERSECTION', na=False).astype(int)
    profile['is_divided_road'] = t.str.contains('DIVIDED', na=False).astype(int)
else:
    profile['is_four_way'] = 0
    profile['is_five_point'] = 0
    profile['is_y_intersection'] = 0
    profile['is_t_intersection'] = 0
    profile['is_divided_road'] = 0

def roadway_context_label(row):
    if row['is_five_point'] == 1 or row['is_four_way'] == 1 or row['is_y_intersection'] == 1:
        return 'COMPLEX_INTERSECTION'
    if row['is_divided_road'] == 1:
        return 'DIVIDED_ROAD'
    return 'OTHER'

profile['roadway_context'] = profile.apply(roadway_context_label, axis=1)

# ------------------------------------------------------------
# Aggregate condition profiles
# NOTE: this is the condition-level risk table
# ------------------------------------------------------------
agg_cols = [
    'grid_lat', 'grid_lon',
    'weather_bucket', 'lighting_bucket',
    'time_bucket', 'speed_context'
]

agg = (
    profile.groupby(agg_cols, as_index=False)
           .agg(
               crash_count=('is_severe', 'size'),
               severe_count=('is_severe', 'sum'),
               severe_rate=('is_severe', 'mean'),
               avg_posted_speed=('POSTED_SPEED_LIMIT', 'mean'),
               pct_intersection=('is_intersection', 'mean'),
               pct_hit_run=('is_hit_and_run', 'mean'),
               pct_self_reported=('is_self_reported', 'mean'),
               pct_night=('is_night', 'mean'),
               pct_weekend=('is_weekend', 'mean'),
               pct_rush_hour=('is_rush_hour', 'mean'),
               pct_divided_road=('is_divided_road', 'mean')
           )
)

agg['severe_rate_pct'] = agg['severe_rate'] * 100

# ------------------------------------------------------------
# Empirical Bayes shrinkage
# ------------------------------------------------------------
global_rate = profile['is_severe'].mean()
alpha = 30

agg['predicted_severe_rate'] = (
    agg['severe_count'] + alpha * global_rate
) / (
    agg['crash_count'] + alpha
)

agg['predicted_severe_rate_pct'] = agg['predicted_severe_rate'] * 100

# Policy-oriented risk score
agg['risk_score'] = agg['predicted_severe_rate'] * np.log1p(agg['crash_count'])

print(f'Total condition profiles: {len(agg):,}')
print(f'Global severe rate: {global_rate*100:.3f}%')

display(
    agg[['crash_count', 'severe_count', 'severe_rate_pct', 'predicted_severe_rate_pct', 'risk_score']].describe()
)


## 2) Clustering workflow (TODO)


In [ ]:
# TODO(C1): Step 4B — Collapse condition risk to grid-cell clustering table

grid_features = (
    agg.groupby(['grid_lat', 'grid_lon'], as_index=False)
       .agg(
           n_profiles=('risk_score', 'size'),
           total_crashes=('crash_count', 'sum'),
           total_severe=('severe_count', 'sum'),
           mean_predicted_risk=('predicted_severe_rate', 'mean'),
           max_predicted_risk=('predicted_severe_rate', 'max'),
           mean_risk_score=('risk_score', 'mean'),
           max_risk_score=('risk_score', 'max'),
           mean_speed=('avg_posted_speed', 'mean'),
           mean_pct_intersection=('pct_intersection', 'mean'),
           mean_pct_hit_run=('pct_hit_run', 'mean'),
           mean_pct_self_reported=('pct_self_reported', 'mean'),
           mean_pct_night=('pct_night', 'mean'),
           mean_pct_weekend=('pct_weekend', 'mean'),
           mean_pct_rush_hour=('pct_rush_hour', 'mean'),
           mean_pct_divided_road=('pct_divided_road', 'mean')
       )
)

grid_features['severe_rate'] = grid_features['total_severe'] / grid_features['total_crashes']
grid_features['severe_rate_pct'] = grid_features['severe_rate'] * 100
grid_features['mean_predicted_risk_pct'] = grid_features['mean_predicted_risk'] * 100
grid_features['max_predicted_risk_pct'] = grid_features['max_predicted_risk'] * 100

print(f'Grid cells for clustering: {len(grid_features):,}')
display(grid_features.head())
display(grid_features.describe())

# KMeans clustering
cluster_features = [
    'n_profiles',
    'total_crashes',
    'total_severe',
    'mean_predicted_risk',
    'max_predicted_risk',
    'mean_risk_score',
    'max_risk_score',
    'mean_speed',
    'mean_pct_intersection',
    'mean_pct_hit_run',
    'mean_pct_self_reported',
    'mean_pct_night',
    'mean_pct_weekend',
    'mean_pct_rush_hour',
    'mean_pct_divided_road'
]

X_cluster = grid_features[cluster_features].copy()
X_cluster = X_cluster.fillna(X_cluster.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# ------------------------------------------------------------
# Choose k with silhouette score
# ------------------------------------------------------------
k_values = range(2, 9)
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sil_scores.append(sil)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), sil_scores, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.title('KMeans: silhouette score by k')
plt.tight_layout()
save_fig('28_silhouette_scores.png')
plt.show()

best_k = list(k_values)[int(np.argmax(sil_scores))]
print(f'Best k by silhouette score: {best_k}')

# Fit final KMeans
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
grid_features['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

print('KMeans cluster counts:')
print(grid_features['kmeans_cluster'].value_counts().sort_index())

kmeans_summary = (
    grid_features.groupby('kmeans_cluster')[cluster_features + ['severe_rate_pct', 'mean_predicted_risk_pct']]
                 .mean()
                 .round(3)
)

display(kmeans_summary)


In [ ]:
# DBSCAN clustering
# Try a few eps values
eps_values = [0.8, 1.0, 1.2, 1.5]
dbscan_results = []

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=10)
    labels = db.fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    dbscan_results.append({
        'eps': eps,
        'n_clusters': n_clusters,
        'n_noise_points': n_noise
    })

dbscan_results = pd.DataFrame(dbscan_results)
display(dbscan_results)

# Choose one eps manually after reviewing the table
chosen_eps = 1.2
dbscan = DBSCAN(eps=chosen_eps, min_samples=10)
grid_features['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

print('DBSCAN cluster counts:')
print(grid_features['dbscan_cluster'].value_counts().sort_index())

dbscan_summary = (
    grid_features.groupby('dbscan_cluster')[cluster_features + ['severe_rate_pct', 'mean_predicted_risk_pct']]
                 .mean()
                 .round(3)
)

display(dbscan_summary)

In [ ]:
#Step 4E — Spatial visualization of clusters

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=grid_features,
    x='grid_lon',
    y='grid_lat',
    hue='kmeans_cluster',
    palette='tab10',
    s=70,
    alpha=0.85
)
plt.title('Spatial clustering of grid cells — KMeans')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='KMeans cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
save_fig('29_kmeans_spatial.png')
plt.show()

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=grid_features,
    x='grid_lon',
    y='grid_lat',
    hue='dbscan_cluster',
    palette='tab10',
    s=70,
    alpha=0.85
)
plt.title('Spatial clustering of grid cells — DBSCAN')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='DBSCAN cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
save_fig('30_dbscan_spatial.png')
plt.show()


## 3) Output contract for dashboard (TODO)


In [ ]:
# Export cluster_labels.csv for notebook 05
high_risk_k = kmeans_summary['mean_predicted_risk_pct'].idxmax()

cluster_labels = grid_features[['grid_lat', 'grid_lon']].copy()
cluster_labels['cluster_kmeans'] = grid_features['kmeans_cluster']
cluster_labels['cluster_dbscan'] = grid_features['dbscan_cluster']
cluster_labels['is_high_risk_cluster'] = (grid_features['kmeans_cluster'] == high_risk_k).astype(int)
cluster_labels['crash_count'] = grid_features['total_crashes']
cluster_labels['severe_rate'] = grid_features['severe_rate']
cluster_labels['avg_risk_score'] = grid_features['mean_risk_score']

cluster_path = CLEAN_DIR / 'cluster_labels.csv'
cluster_labels.to_csv(cluster_path, index=False)

required_cluster_columns = [
    'grid_lat', 'grid_lon',
    'cluster_kmeans', 'cluster_dbscan',
    'is_high_risk_cluster',
    'crash_count', 'severe_rate',
    'avg_risk_score'
]

print('cluster_labels required columns:', required_cluster_columns)
print('Saved:', cluster_path.resolve())
display(cluster_labels.head())


## 5) Spatial Dashboard & Policy Integration

## Alignment Notes
- **Objective**: translate crash risk outputs into a dashboard artifact for policy prioritization and stakeholder communication.
- **Primary question**: where should CDOT prioritize interventions based on spatial risk concentration and condition patterns?
- **Input data**:
  - `dataset/cleaned_data/risk_scores.csv` (condition-profile risk scoring output from Notebook 03)
  - `dataset/cleaned_data/train.csv` and `dataset/cleaned_data/test.csv` (full crash-level context for descriptive analysis)
  - `dataset/cleaned_data/cluster_labels.csv` (from Notebook 04)
- **Process in this notebook**:
  1. load and validate continuity from prior notebooks,
  2. build dashboard-ready aggregations,
  3. export static figures,
  4. generate a standalone HTML dashboard,
  5. export policy-ready CSV tables.
- **Output artifacts**:
  - `image/05_dashboard/chicago_crash_risk_dashboard.html`
  - `image/05_dashboard/*.png` (4 figures)
  - `dataset/cleaned_data/policy_priority_cells.csv`
  - `dataset/cleaned_data/dashboard_risk_cluster_table.csv`

This notebook uses the real cluster output from Notebook 04.

In [ ]:
import json
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from folium import plugins
from IPython.display import Markdown, display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_ROOT = Path('dataset') if Path('dataset').exists() else Path('../dataset')
CLEAN_DIR = DATA_ROOT / 'cleaned_data'
IMAGE_DIR = MAIN_IMAGE_DIR
REFERENCE_DIR = DATA_ROOT / 'reference'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_DATA_AVAILABLE = True
RANDOM_STATE = 42


def save_fig(name: str):
    path = IMAGE_DIR / name
    plt.savefig(path, bbox_inches='tight', dpi=160)
    return path

print(f'Using data: {CLEAN_DIR.resolve()}')
print(f'Using image output: {IMAGE_DIR.resolve()}')
print(f'Using reference data: {REFERENCE_DIR.resolve()}')


## 1) Data Loading and Continuity Checks

In [ ]:
risk_scores = pd.read_csv(CLEAN_DIR / 'risk_scores.csv')
train_df = pd.read_csv(CLEAN_DIR / 'train.csv')
test_df = pd.read_csv(CLEAN_DIR / 'test.csv')
all_crashes = pd.concat([train_df, test_df], ignore_index=True)

required_risk_cols = {
    'grid_lat', 'grid_lon', 'weather_bucket', 'lighting_bucket', 'time_bucket', 'speed_bucket',
    'crash_count', 'severe_count', 'severe_rate', 'predicted_severe_rate', 'risk_score', 'rank', 'recommendation'
}
required_crash_cols = {
    'CRASH_DATE', 'LATITUDE', 'LONGITUDE', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'is_severe'
}

missing_risk = required_risk_cols - set(risk_scores.columns)
missing_train = required_crash_cols - set(train_df.columns)
missing_test = required_crash_cols - set(test_df.columns)
missing_all = required_crash_cols - set(all_crashes.columns)
assert not missing_risk, f'Missing risk columns: {missing_risk}'
assert not missing_train, f'Missing train columns: {missing_train}'
assert not missing_test, f'Missing test columns: {missing_test}'
assert not missing_all, f'Missing combined columns: {missing_all}'

all_crashes['CRASH_DATE'] = pd.to_datetime(all_crashes['CRASH_DATE'], errors='coerce')

print(f'risk_scores shape: {risk_scores.shape}')
print(f'train shape: {train_df.shape}')
print(f'test shape: {test_df.shape}')
print(f'all_crashes shape: {all_crashes.shape}')
print(f'risk_score range: {risk_scores.risk_score.min():.4f} to {risk_scores.risk_score.max():.4f}')

In [ ]:
grid_cells = (
    risk_scores
    .groupby(['grid_lat', 'grid_lon'], as_index=False)
    .agg(
        risk_score=('risk_score', 'mean'),
        max_risk_score=('risk_score', 'max'),
        crash_count=('crash_count', 'sum'),
        severe_count=('severe_count', 'sum')
    )
)
grid_cells['severe_rate'] = np.where(grid_cells['crash_count'] > 0, grid_cells['severe_count'] / grid_cells['crash_count'], 0.0)

assert CLUSTER_DATA_AVAILABLE, 'Notebook 05 requires real cluster labels from notebook 04.'
cluster_df = pd.read_csv(CLEAN_DIR / 'cluster_labels.csv')
expected = {'grid_lat', 'grid_lon', 'cluster_kmeans', 'cluster_dbscan', 'is_high_risk_cluster'}
assert expected.issubset(cluster_df.columns), f'cluster_labels.csv needs columns: {expected}'

grid_cells = grid_cells.merge(
    cluster_df[['grid_lat', 'grid_lon', 'cluster_kmeans', 'cluster_dbscan', 'is_high_risk_cluster']],
    on=['grid_lat', 'grid_lon'],
    how='left'
)

missing_clusters = grid_cells['cluster_kmeans'].isna().sum()
assert missing_clusters == 0, f'Missing cluster labels for {missing_clusters} grid cells.'

grid_cells['cluster_kmeans'] = grid_cells['cluster_kmeans'].astype(int)
grid_cells['cluster_dbscan'] = grid_cells['cluster_dbscan'].astype(int)
grid_cells['is_high_risk_cluster'] = grid_cells['is_high_risk_cluster'].astype(int)

print(f'Unique grid cells: {len(grid_cells):,}')
print('Cluster mode: notebook 04 exported clusters')


### 1.1 Neighborhood Context for Real-Map Exploration

This subsection adds Chicago community-area boundaries so crash patterns can be interpreted using real neighborhood names, not only latitude and longitude.

In [ ]:
community_geojson_path = REFERENCE_DIR / 'chicago_community_areas.geojson'

if not community_geojson_path.exists():
    geojson_urls = [
        'https://data.cityofchicago.org/resource/igwz-8jzy.geojson?$limit=5000',
    ]
    downloaded = False
    for url in geojson_urls:
        try:
            with urlopen(url, timeout=30) as resp:
                payload = resp.read()
            community_geojson_path.write_bytes(payload)
            downloaded = True
            break
        except URLError:
            continue
    assert downloaded, 'Could not download Chicago community-area GeoJSON. Please check network or provide local file.'

community_areas = gpd.read_file(community_geojson_path).to_crs(epsg=4326)
community_areas['community'] = community_areas['community'].str.title()
community_areas = community_areas[['community', 'geometry']].copy()

community_areas_simple = community_areas.copy()
community_areas_simple['geometry'] = community_areas_simple['geometry'].simplify(0.00025, preserve_topology=True)

display(community_areas.head(3))

In [ ]:
grid_points = gpd.GeoDataFrame(
    grid_cells[['grid_lat', 'grid_lon']].copy(),
    geometry=gpd.points_from_xy(grid_cells['grid_lon'], grid_cells['grid_lat']),
    crs='EPSG:4326'
)

grid_with_community = gpd.sjoin(
    grid_points,
    community_areas[['community', 'geometry']],
    how='left',
    predicate='within'
).drop(columns=['index_right'])

grid_cells = grid_cells.merge(
    grid_with_community[['grid_lat', 'grid_lon', 'community']],
    on=['grid_lat', 'grid_lon'],
    how='left'
)
grid_cells['community'] = grid_cells['community'].fillna('Unassigned')

risk_scores = risk_scores.merge(
    grid_cells[['grid_lat', 'grid_lon', 'community']],
    on=['grid_lat', 'grid_lon'],
    how='left'
)
risk_scores['community'] = risk_scores['community'].fillna('Unassigned')

neighborhood_stats = (
    risk_scores
    .groupby('community', as_index=False)
    .agg(
        crash_count=('crash_count', 'sum'),
        severe_count=('severe_count', 'sum'),
        avg_risk_score=('risk_score', 'mean'),
        profile_count=('risk_score', 'size')
    )
)
neighborhood_stats['severe_rate_pct'] = np.where(
    neighborhood_stats['crash_count'] > 0,
    100 * neighborhood_stats['severe_count'] / neighborhood_stats['crash_count'],
    np.nan
)
neighborhood_stats = neighborhood_stats.sort_values('crash_count', ascending=False)

neighborhood_stats.head(12)

In [ ]:
top3_crash = neighborhood_stats.head(3)
top3_severe = neighborhood_stats.sort_values('severe_rate_pct', ascending=False).head(3)

insight_text = (
    f"**Descriptive Neighborhood Insights (No ML):**\n\n"
    f"- Crash volume is concentrated in a small set of neighborhoods. Top 3 by crash count are "
    f"**{top3_crash.iloc[0]['community']}**, **{top3_crash.iloc[1]['community']}**, and **{top3_crash.iloc[2]['community']}**.\n"
    f"- Severe-rate hotspots are different from volume hotspots. Top 3 by severe rate are "
    f"**{top3_severe.iloc[0]['community']}**, **{top3_severe.iloc[1]['community']}**, and **{top3_severe.iloc[2]['community']}**.\n"
    f"- Operationally, this supports two lenses: high-volume neighborhoods for broad safety management, "
    f"and high-severity neighborhoods for targeted intervention design."
)

display(Markdown(insight_text))
display(neighborhood_stats.head(15))

In [ ]:
map_center = [float(all_crashes['LATITUDE'].mean()), float(all_crashes['LONGITUDE'].mean())]

context_map = folium.Map(
    location=map_center,
    zoom_start=10,
    tiles='CartoDB positron',
    control_scale=True
)

community_map_df = community_areas_simple.merge(neighborhood_stats, on='community', how='left')
community_map_df['crash_count'] = community_map_df['crash_count'].fillna(0).astype(int)
community_map_df['severe_rate_pct'] = community_map_df['severe_rate_pct'].fillna(0.0)
community_map_df['avg_risk_score'] = community_map_df['avg_risk_score'].fillna(0.0)

folium.Choropleth(
    geo_data=community_map_df.to_json(),
    data=community_map_df,
    columns=['community', 'crash_count'],
    key_on='feature.properties.community',
    fill_color='YlOrRd',
    fill_opacity=0.65,
    line_opacity=0.35,
    nan_fill_color='#f2f2f2',
    legend_name='Crash count by neighborhood (aggregated profiles)'
).add_to(context_map)

folium.GeoJson(
    community_map_df,
    name='Neighborhood boundary and stats',
    style_function=lambda _: {'fillOpacity': 0.0, 'weight': 0.8, 'color': '#666'},
    tooltip=folium.GeoJsonTooltip(
        fields=['community', 'crash_count', 'severe_rate_pct', 'avg_risk_score'],
        aliases=['Neighborhood', 'Crash count', 'Severe rate (%)', 'Avg risk score'],
        localize=True,
        sticky=False,
        labels=True,
    ),
).add_to(context_map)

# Derive tier from recommendation for map filtering, independent from later cells.
def recommendation_to_tier_local(rec: str) -> str:
    if isinstance(rec, str):
        if rec.startswith('PRIORITY'):
            return 'PRIORITY'
        if rec.startswith('HIGH'):
            return 'HIGH'
        if rec.startswith('MODERATE'):
            return 'MODERATE'
    return 'STANDARD'


def mode_or_unknown(series: pd.Series) -> str:
    mode = series.dropna().mode()
    if len(mode) > 0:
        return str(mode.iloc[0])
    return 'UNKNOWN'


map_points = risk_scores[['grid_lat', 'grid_lon', 'community', 'recommendation', 'crash_count', 'severe_count', 'risk_score']].copy()
map_points['tier'] = map_points['recommendation'].map(recommendation_to_tier_local)
map_points = (
    map_points
    .groupby(['grid_lat', 'grid_lon', 'community', 'tier'], as_index=False)
    .agg(
        crash_count=('crash_count', 'sum'),
        severe_count=('severe_count', 'sum'),
        risk_score=('risk_score', 'mean')
    )
)
map_points['severe_rate_pct'] = np.where(map_points['crash_count'] > 0, 100 * map_points['severe_count'] / map_points['crash_count'], 0.0)

crash_geo_context = (
    all_crashes[['LATITUDE', 'LONGITUDE', 'TRAFFICWAY_TYPE']]
    .dropna(subset=['LATITUDE', 'LONGITUDE'])
    .assign(
        grid_lat=lambda d: d['LATITUDE'].round(2),
        grid_lon=lambda d: d['LONGITUDE'].round(2)
    )
)

point_context = (
    crash_geo_context
    .groupby(['grid_lat', 'grid_lon'], as_index=False)
    .agg(dominant_trafficway=('TRAFFICWAY_TYPE', mode_or_unknown))
)

map_points = map_points.merge(point_context, on=['grid_lat', 'grid_lon'], how='left')
map_points['dominant_trafficway'] = map_points['dominant_trafficway'].fillna('UNKNOWN')
map_points['location_label'] = (
    map_points['dominant_trafficway'].str.replace('_', ' ', regex=False).str.title()
    + ' | approx ('
    + map_points['grid_lat'].round(4).astype(str)
    + ', '
    + map_points['grid_lon'].round(4).astype(str)
    + ')'
)

priority_high = map_points[map_points['tier'].isin(['PRIORITY', 'HIGH'])]
others = map_points[~map_points['tier'].isin(['PRIORITY', 'HIGH'])]
map_points_sample = pd.concat([
    priority_high,
    others.sample(n=min(1200, len(others)), random_state=RANDOM_STATE, weights=others['crash_count'].clip(lower=1))
], ignore_index=True)

tier_color = {
    'PRIORITY': '#dc2626',
    'HIGH': '#ea580c',
    'MODERATE': '#ca8a04',
    'STANDARD': '#16a34a'
}

for tier_name in ['PRIORITY', 'HIGH', 'MODERATE', 'STANDARD']:
    fg = folium.FeatureGroup(name=f'{tier_name} grid cells', show=(tier_name in ['PRIORITY', 'HIGH']))
    sub = map_points_sample[map_points_sample['tier'] == tier_name]
    for r in sub.itertuples(index=False):
        folium.CircleMarker(
            location=[r.grid_lat, r.grid_lon],
            radius=max(2, min(7, np.sqrt(r.crash_count))),
            color=tier_color[tier_name],
            fill=True,
            fill_opacity=0.55,
            weight=0.8,
            popup=(
                f"Neighborhood: {r.community}<br>"
                f"Tier: {r.tier}<br>"
                f"Crashes: {int(r.crash_count)}<br>"
                f"Severe rate: {r.severe_rate_pct:.2f}%<br>"
                f"Road context: {str(r.dominant_trafficway).replace('_', ' ').title()}<br>"
                f"Approx point: ({r.grid_lat:.4f}, {r.grid_lon:.4f})<br>"
                f"Risk score: {r.risk_score:.4f}"
            )
        ).add_to(fg)
    fg.add_to(context_map)

search_layer = folium.GeoJson(
    community_map_df[['community', 'geometry']].to_json(),
    name='Neighborhood search source',
    show=False,
)
search_layer.add_to(context_map)

plugins.Search(
    layer=search_layer,
    geom_type='Polygon',
    search_label='community',
    placeholder='Search neighborhood',
    collapsed=False,
).add_to(context_map)

plugins.Fullscreen(position='topright').add_to(context_map)
plugins.MiniMap(toggle_display=True).add_to(context_map)
folium.LayerControl(collapsed=False).add_to(context_map)

context_map

## 2) Dashboard Aggregations

In [ ]:
kpis = {
    'total_crashes': int(all_crashes.shape[0]),
    'severe_rate_pct': float(all_crashes['is_severe'].mean() * 100),
    'high_risk_zones': int(risk_scores.loc[risk_scores['rank'] <= 200, ['grid_lat', 'grid_lon']].drop_duplicates().shape[0]),
    'max_risk_score': float(risk_scores['risk_score'].max())
}

kpi_df = pd.DataFrame([kpis])
kpi_df

In [ ]:
tier_labels = ['STANDARD', 'MODERATE', 'HIGH', 'PRIORITY']
grid_cells['tier'] = pd.qcut(
    grid_cells['risk_score'].rank(method='first'),
    q=4,
    labels=tier_labels
).astype(str)

priority_high = grid_cells[grid_cells['tier'].isin(['PRIORITY', 'HIGH'])].copy()
rest = grid_cells[~grid_cells['tier'].isin(['PRIORITY', 'HIGH'])].copy()

remaining_n = max(0, 800 - len(priority_high))
if remaining_n > 0 and len(rest) > 0:
    rest_sample = rest.sample(
        n=min(remaining_n, len(rest)),
        random_state=RANDOM_STATE,
        weights=rest['crash_count'].clip(lower=1)
    )
else:
    rest_sample = rest.head(0)

scatter_df = pd.concat([priority_high, rest_sample], ignore_index=True).drop_duplicates(['grid_lat', 'grid_lon'])
scatter_df = scatter_df.rename(columns={'grid_lat': 'lat', 'grid_lon': 'lon'})

scatter_df[['lat', 'lon', 'risk_score', 'crash_count', 'tier']].head()

In [ ]:
wl_agg = (
    risk_scores
    .groupby(['weather_bucket', 'lighting_bucket'], as_index=False)
    .agg(crash_count=('crash_count', 'sum'), severe_count=('severe_count', 'sum'))
)
wl_agg['severe_rate_pct'] = np.where(wl_agg['crash_count'] > 0, 100 * wl_agg['severe_count'] / wl_agg['crash_count'], np.nan)

weather_order = sorted(wl_agg['weather_bucket'].dropna().unique())
lighting_order = sorted(wl_agg['lighting_bucket'].dropna().unique())

wl_pivot = wl_agg.pivot(index='weather_bucket', columns='lighting_bucket', values='severe_rate_pct').reindex(index=weather_order, columns=lighting_order)
wl_pivot

In [ ]:
day_map = {
    1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed', 5: 'Thu', 6: 'Fri', 7: 'Sat'
}

temporal = (
    all_crashes
    .dropna(subset=['CRASH_DAY_OF_WEEK', 'CRASH_HOUR'])
    .assign(day_label=lambda d: d['CRASH_DAY_OF_WEEK'].astype(int).map(day_map))
    .groupby(['day_label', 'CRASH_HOUR'], as_index=False)['is_severe'].mean()
)
temporal['severe_rate_pct'] = temporal['is_severe'] * 100

day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
hour_order = list(range(24))

temporal_pivot = temporal.pivot(index='day_label', columns='CRASH_HOUR', values='severe_rate_pct').reindex(index=day_order, columns=hour_order)
temporal_pivot.head()

In [ ]:
def build_condition_severe_table(group_col: str):
    tab = (
        risk_scores
        .groupby(group_col, as_index=False)
        .agg(
            crash_count=('crash_count', 'sum'),
            severe_count=('severe_count', 'sum')
        )
    )
    tab['severe_rate_pct'] = np.where(
        tab['crash_count'] > 0,
        100 * tab['severe_count'] / tab['crash_count'],
        0.0
    )
    return tab.sort_values(['severe_rate_pct', 'crash_count'], ascending=[False, False])

cond_weather = build_condition_severe_table('weather_bucket')
cond_light = build_condition_severe_table('lighting_bucket')
cond_time = build_condition_severe_table('time_bucket')
cond_speed = build_condition_severe_table('speed_bucket')

condition_tables = {
    'weather': cond_weather,
    'lighting': cond_light,
    'time': cond_time,
    'speed': cond_speed,
}

for name, tab in condition_tables.items():
    print(name, 'rows:', len(tab))

cond_weather[['weather_bucket', 'crash_count', 'severe_count', 'severe_rate_pct']].head()

In [ ]:
def recommendation_to_tier(rec: str) -> str:
    if isinstance(rec, str):
        if rec.startswith('PRIORITY'):
            return 'PRIORITY'
        if rec.startswith('HIGH'):
            return 'HIGH'
        if rec.startswith('MODERATE'):
            return 'MODERATE'
    return 'STANDARD'

risk_scores['tier'] = risk_scores['recommendation'].map(recommendation_to_tier)

neighborhood_tier_stats = (
    risk_scores
    .groupby(['tier', 'community'], as_index=False)
    .agg(
        crash_count=('crash_count', 'sum'),
        severe_count=('severe_count', 'sum'),
        avg_risk_score=('risk_score', 'mean')
    )
)
neighborhood_tier_stats['severe_rate_pct'] = np.where(
    neighborhood_tier_stats['crash_count'] > 0,
    100 * neighborhood_tier_stats['severe_count'] / neighborhood_tier_stats['crash_count'],
    0.0
)

dominant_tier = (
    neighborhood_tier_stats
    .sort_values(['community', 'severe_rate_pct', 'crash_count'], ascending=[True, False, False])
    .drop_duplicates('community')
    [['community', 'tier']]
    .rename(columns={'tier': 'dominant_tier'})
)

neighborhood_severe_rank = (
    neighborhood_stats
    .merge(dominant_tier, on='community', how='left')
    .assign(dominant_tier=lambda d: d['dominant_tier'].fillna('STANDARD'))
    .sort_values(['severe_rate_pct', 'crash_count'], ascending=[False, False])
    [['community', 'crash_count', 'severe_count', 'severe_rate_pct', 'avg_risk_score', 'dominant_tier']]
)

sankey_df = (
    risk_scores
    .groupby(['tier', 'recommendation'], as_index=False)
    .agg(value=('crash_count', 'sum'))
)

threshold_rows = []
total_severe = max(risk_scores['severe_count'].sum(), 1)
for cutoff in [50, 100, 200, 500, 1000, 2000, 5000]:
    sub = risk_scores[risk_scores['rank'] <= cutoff]
    threshold_rows.append({
        'rank_cutoff': cutoff,
        'zone_count': int(sub[['grid_lat', 'grid_lon']].drop_duplicates().shape[0]),
        'severe_capture_pct': float(100 * sub['severe_count'].sum() / total_severe)
    })
threshold_df = pd.DataFrame(threshold_rows)

risk_vals = risk_scores['risk_score'].dropna().to_numpy()
if len(risk_vals) > 1:
    x_grid = np.linspace(risk_vals.min(), risk_vals.max(), 140)
    std = np.std(risk_vals, ddof=1)
    bw = 1.06 * std * (len(risk_vals) ** (-1/5)) if std > 0 else max((risk_vals.max() - risk_vals.min()) / 30, 1e-4)
    z = (x_grid[:, None] - risk_vals[None, :]) / max(bw, 1e-6)
    density = np.exp(-0.5 * z * z).sum(axis=1) / (len(risk_vals) * max(bw, 1e-6) * np.sqrt(2 * np.pi))
else:
    x_grid = np.array([risk_vals[0] if len(risk_vals) == 1 else 0.0])
    density = np.array([1.0])

hist_df = pd.DataFrame({
    'x': x_grid,
    'density': density
})

top_profiles = risk_scores.sort_values('rank').head(50).copy()
top_profiles['severe_rate_pct'] = top_profiles['severe_rate'] * 100

tier_cards = (
    grid_cells
    .groupby('tier', as_index=False)
    .agg(cell_count=('tier', 'count'), avg_risk=('risk_score', 'mean'), crash_count=('crash_count', 'sum'))
    .set_index('tier')
    .reindex(['PRIORITY', 'HIGH', 'MODERATE', 'STANDARD'])
    .reset_index()
)

threshold_df

# Daily descriptive trend for dashboard timeline (EDA-first context)
daily_trend = (
    all_crashes.dropna(subset=['CRASH_DATE'])
    .assign(crash_day=lambda d: d['CRASH_DATE'].dt.floor('D'))
    .groupby('crash_day', as_index=False)
    .agg(
        total_crashes=('is_severe', 'size'),
        severe_crashes=('is_severe', 'sum')
    )
    .sort_values('crash_day')
)
daily_trend['severe_rate_pct'] = np.where(
    daily_trend['total_crashes'] > 0,
    100 * daily_trend['severe_crashes'] / daily_trend['total_crashes'],
    0.0
)

# Detail tables for polygon cross-filter in timeline and weather views.
grid_context_lookup = grid_cells[['grid_lat', 'grid_lon', 'community', 'tier']].drop_duplicates()

crash_with_context = (
    all_crashes
    .dropna(subset=['CRASH_DATE', 'LATITUDE', 'LONGITUDE'])
    .assign(
        grid_lat=lambda d: d['LATITUDE'].round(2),
        grid_lon=lambda d: d['LONGITUDE'].round(2),
        crash_day=lambda d: d['CRASH_DATE'].dt.floor('D')
    )
    .merge(grid_context_lookup, on=['grid_lat', 'grid_lon'], how='left')
)
crash_with_context['community'] = crash_with_context['community'].fillna('Unassigned')
crash_with_context['tier'] = crash_with_context['tier'].fillna('STANDARD')

timeline_detail = (
    crash_with_context
    .groupby(['crash_day', 'community', 'tier'], as_index=False)
    .agg(
        total_crashes=('is_severe', 'size'),
        severe_crashes=('is_severe', 'sum')
    )
)

wl_detail = (
    risk_scores
    .groupby(['community', 'tier', 'weather_bucket', 'lighting_bucket'], as_index=False)
    .agg(
        crash_count=('crash_count', 'sum'),
        severe_count=('severe_count', 'sum')
    )
)
wl_detail['severe_rate_pct'] = np.where(
    wl_detail['crash_count'] > 0,
    100 * wl_detail['severe_count'] / wl_detail['crash_count'],
    0.0
)

eligible_severe = neighborhood_stats[neighborhood_stats['crash_count'] >= 100].sort_values('severe_rate_pct', ascending=False)
insight_summary = {
    'top_volume': neighborhood_stats.head(3)['community'].tolist(),
    'top_severe_rate': eligible_severe.head(3)['community'].tolist(),
    'peak_day': str(daily_trend.loc[daily_trend['total_crashes'].idxmax(), 'crash_day'].date()),
    'peak_crashes': int(daily_trend['total_crashes'].max()),
    'daily_avg_crashes': float(daily_trend['total_crashes'].mean())
}


## 3) Matplotlib Visualizations (saved to `image/05_dashboard/`)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(wl_pivot, cmap='YlOrRd', annot=True, fmt='.2f', linewidths=0.4, ax=ax)
ax.set_title('Severe Rate (%) by Weather x Lighting')
ax.set_xlabel('Lighting Bucket')
ax.set_ylabel('Weather Bucket')
path_2 = save_fig('31_weather_lighting_heatmap.png')
plt.show()
path_2

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(temporal_pivot, cmap='Reds', linewidths=0.2, ax=ax)
ax.set_title('Severe Rate (%) by Day and Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
path_3 = save_fig('32_temporal_heatmap.png')
plt.show()
path_3

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_df['x'], hist_df['density'], color='#1d4ed8', linewidth=2)
ax.fill_between(hist_df['x'], hist_df['density'], color='#93c5fd', alpha=0.35)
ax.set_title('Risk Score Density')
ax.set_xlabel('Risk score')
ax.set_ylabel('Density')
ax.grid(alpha=0.2)
path_4 = save_fig('33_risk_distribution.png')
plt.show()
path_4

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(threshold_df['rank_cutoff'], threshold_df['zone_count'], marker='o', label='Zones captured', color='#1d4ed8')
ax2 = ax.twinx()
ax2.plot(threshold_df['rank_cutoff'], threshold_df['severe_capture_pct'], marker='s', linestyle='--',
         label='Severe capture (%)', color='#dc2626')

ax.set_title('Policy Threshold Tradeoff')
ax.set_xlabel('Rank cutoff')
ax.set_ylabel('Zone count', color='#1d4ed8')
ax2.set_ylabel('Severe capture (%)', color='#dc2626')
ax.grid(alpha=0.2)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

path_5 = save_fig('34_policy_threshold.png')
plt.show()
path_5

## 4) Static HTML Dashboard Generation

In [ ]:
# Compact JSON blobs for embedding

wl_blob = json.dumps({
    'rows': list(wl_pivot.index),
    'cols': list(wl_pivot.columns),
    'vals': [[None if pd.isna(v) else float(v) for v in row] for row in wl_pivot.values]
}, separators=(',', ':'))

temporal_blob = json.dumps({
    'days': list(temporal_pivot.index),
    'hours': [int(h) for h in temporal_pivot.columns],
    'vals': [[None if pd.isna(v) else float(v) for v in row] for row in temporal_pivot.values]
}, separators=(',', ':'))

condition_blob = json.dumps({
    'weather': cond_weather.to_dict('records'),
    'lighting': cond_light.to_dict('records'),
    'time': cond_time.to_dict('records'),
    'speed': cond_speed.to_dict('records')
}, separators=(',', ':'))

sankey_blob = json.dumps(sankey_df.to_dict('records'), separators=(',', ':'))
threshold_blob = json.dumps(threshold_df.to_dict('records'), separators=(',', ':'))
hist_blob = json.dumps(hist_df.to_dict('records'), separators=(',', ':'))
top_blob = json.dumps(top_profiles[['rank', 'grid_lat', 'grid_lon', 'community', 'weather_bucket', 'lighting_bucket', 'time_bucket', 'speed_bucket', 'crash_count', 'severe_rate_pct', 'risk_score', 'recommendation']].to_dict('records'), separators=(',', ':'))
neighborhood_blob = json.dumps(
    neighborhood_severe_rank.to_dict('records'),
    separators=(',', ':')
)
map_points_blob = json.dumps([
    {
        'la': round(float(r.grid_lat), 4),
        'lo': round(float(r.grid_lon), 4),
        'n': r.community,
        't': r.tier,
        'c': int(r.crash_count),
        's': round(float(r.severe_rate_pct), 2),
        'tw': str(r.dominant_trafficway),
        'loc': r.location_label
    }
    for r in map_points_sample.itertuples(index=False)
], separators=(',', ':'))

daily_blob = json.dumps([
    {
        'd': str(r.crash_day.date()),
        'c': int(r.total_crashes),
        's': int(r.severe_crashes),
        'sr': round(float(r.severe_rate_pct), 3)
    }
    for r in daily_trend.itertuples(index=False)
], separators=(',', ':'))

daily_detail_blob = json.dumps([
    {
        'd': str(r.crash_day.date()),
        'n': r.community,
        't': r.tier,
        'c': int(r.total_crashes),
        's': int(r.severe_crashes)
    }
    for r in timeline_detail.itertuples(index=False)
], separators=(',', ':'))

wl_detail_blob = json.dumps([
    {
        'n': r.community,
        't': r.tier,
        'w': r.weather_bucket,
        'l': r.lighting_bucket,
        'c': int(r.crash_count),
        's': int(r.severe_count),
        'sr': round(float(r.severe_rate_pct), 3)
    }
    for r in wl_detail.itertuples(index=False)
], separators=(',', ':'))

neigh_geo_blob = community_map_df[['community', 'crash_count', 'severe_rate_pct', 'avg_risk_score', 'geometry']].to_json()
insight_blob = json.dumps(insight_summary, separators=(',', ':'))

kpi_blob = json.dumps(kpis, separators=(',', ':'))
tier_blob = json.dumps(tier_cards.to_dict('records'), separators=(',', ':'))

for name, blob in [
    ('weather_lighting', wl_blob), ('temporal', temporal_blob),
    ('condition', condition_blob), ('sankey', sankey_blob), ('threshold', threshold_blob),
    ('hist', hist_blob), ('top', top_blob), ('neighborhood', neighborhood_blob),
    ('map_points', map_points_blob), ('daily', daily_blob), ('daily_detail', daily_detail_blob),
    ('wl_detail', wl_detail_blob), ('neigh_geo', neigh_geo_blob),
    ('insight', insight_blob), ('kpi', kpi_blob), ('tier', tier_blob)
]:
    print(f'{name:16s}: {len(blob)/1024:7.2f} KB')


In [ ]:
template = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Chicago Crash Risk Dashboard</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2"></script>
  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <script src="https://cdn.jsdelivr.net/npm/d3-sankey@0.12.3"></script>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=" crossorigin=""/>
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js" integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>
  <style>
    :root {
      --bg:#f4f7fb; --card:#ffffff; --border:#d6e0eb; --border-dark:#b8c7d8;
      --text:#1f2937; --text-muted:#5b6b80; --shadow:rgba(15,23,42,0.08);
      --tableau-blue:#4e79a7; --tableau-orange:#f28e2b; --tableau-red:#e15759; --tableau-teal:#76b7b2;
      --tableau-green:#59a14f; --tableau-yellow:#edc948; --tableau-purple:#b07aa1; --tableau-grey:#bab0ab;
      --critical:#d62728; --high:#ff7f0e; --moderate:#bcbd22; --standard:#2ca02c;
    }
    * { box-sizing:border-box; margin:0; padding:0; }
    body { font-family:'Segoe UI',Tahoma,Arial,sans-serif; background:var(--bg); color:var(--text); font-size:13px; }
    .header { background:linear-gradient(180deg,#eef3fa 0%,#e2eaf5 100%); border-bottom:1px solid var(--border-dark); padding:14px 24px; }
    .header-title { font-size:18px; font-weight:700; color:#1d3557; }
    .header-subtitle { font-size:12px; color:#44566c; margin-top:2px; }
    .dashboard { max-width:1600px; margin:0 auto; padding:18px; }
    .row { display:flex; gap:10px; margin-bottom:8px; align-items:stretch; }
    .col { flex:1; min-width:0; } .col-2 { flex:2; min-width:0; }
    @media (max-width: 1180px) { .row { flex-wrap:wrap; } .col, .col-2 { flex:1 1 100%; } }
    .panel {
      --panel-accent: var(--tableau-blue);
      background:var(--card); border:1px solid var(--border); border-radius:10px;
      border-top:4px solid var(--panel-accent); box-shadow:0 2px 10px var(--shadow); overflow:hidden;
      display:flex; flex-direction:column;
    }
    .panel-header {
      padding:10px 12px; background:#f8fbff; border-bottom:1px solid #dbe6f2;
      display:flex; align-items:center;
    }
    .panel-title { font-weight:700; font-size:12px; color:#28445f; letter-spacing:0.01em; }
    .panel-body { padding:12px; overflow:hidden; }
    .map-panel .panel-body { overflow:hidden; }
    .panel.tone-blue { --panel-accent: var(--tableau-blue); }
    .panel.tone-orange { --panel-accent: var(--tableau-orange); }
    .panel.tone-red { --panel-accent: var(--tableau-red); }
    .panel.tone-teal { --panel-accent: var(--tableau-teal); }
    .panel.tone-green { --panel-accent: var(--tableau-green); }
    .panel.tone-purple { --panel-accent: var(--tableau-purple); }
    .panel.tone-grey { --panel-accent: #8a9aab; }

    .kpi-grid { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:10px; margin-bottom:10px; }
    @media (max-width: 980px) { .kpi-grid { grid-template-columns:repeat(2,minmax(0,1fr)); } }
    .kpi-card { background:var(--card); border:1px solid var(--border); border-radius:10px; padding:14px; box-shadow:0 1px 6px var(--shadow); }
    .kpi-label { font-size:10px; text-transform:uppercase; color:var(--text-muted); font-weight:700; }
    .kpi-value { font-size:28px; font-weight:700; margin-top:4px; color:#1d3557; }
    .kpi-detail { font-size:11px; color:#4a5a6e; margin-top:4px; }

    .tier-grid { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:10px; margin-bottom:10px; }
    @media (max-width: 1100px) { .tier-grid { grid-template-columns:repeat(2,minmax(0,1fr)); } }
    .tier-card { border-radius:10px; border:1px solid var(--border); background:var(--card); padding:10px; }
    .tier-card h4 { font-size:12px; margin-bottom:6px; }
    .tier-card .big { font-size:20px; font-weight:700; }
    .tag { font-size:10px; display:inline-block; margin-top:6px; padding:2px 6px; border-radius:999px; color:white; }

    .table-wrap { overflow:auto; max-height:380px; }
    table { width:100%; border-collapse:collapse; font-size:11px; }
    th, td { border-bottom:1px solid #edf2f7; padding:6px; text-align:left; white-space:nowrap; }
    th { position:sticky; top:0; background:#f1f6fc; z-index:1; }

    .heatgrid { display:grid; gap:1px; background:#d5dee8; border:1px solid #d5dee8; }
    .cell { background:white; padding:4px; text-align:center; font-size:10px; }

    .footer { margin-top:12px; font-size:11px; color:#4f6278; }
    .small-bars .grp { margin-bottom:8px; }
    .small-bars .lbl { font-size:10px; margin-bottom:2px; color:#3b4f66; }
    .small-bars .bar { height:8px; background:#e6edf5; border-radius:999px; overflow:hidden; }
    .small-bars .fill { height:100%; background:var(--tableau-blue); }
    .cond-grid { display:grid; grid-template-columns:repeat(4, minmax(0,1fr)); gap:10px; }
    .cond-col { border:1px solid #d9e4ef; border-radius:8px; padding:8px; background:#fcfdff; }
    .cond-head { font-size:11px; font-weight:700; margin-bottom:8px; text-transform:capitalize; }
    @media (max-width: 1100px) { .cond-grid { grid-template-columns:repeat(2, minmax(0,1fr)); } }

    #leafletMap { height: 430px; border:1px solid #d5dee8; border-radius:8px; }
    .filters { display:flex; gap:8px; margin-bottom:8px; flex-wrap:wrap; }
    .filters label { font-size:11px; color:#44566c; }
    .filters select { padding:4px 6px; border:1px solid #c7d5e4; background:#fff; border-radius:6px; font-size:11px; }
    .selected-chip { font-size:11px; color:#36506a; background:#ecf3fb; border:1px solid #c9d9ec; border-radius:999px; padding:4px 10px; }
    .ghost-btn { font-size:11px; background:#fff; border:1px solid #c7d5e4; color:#35506a; border-radius:8px; padding:4px 10px; cursor:pointer; }
    .ghost-btn:hover { background:#eef5fc; }

    canvas { display:block; width:100% !important; }
    #dailyChart, #histChart, #thresholdChart { height:260px !important; }
    .weather-panel .panel-body { height:520px; display:flex; flex-direction:column; }
    .neigh-panel .panel-body { height:520px; display:grid; grid-template-rows:auto 1fr; gap:6px; }
    #wlHeatmap { flex:1; min-height:0; overflow:auto; }
    .neigh-scroll { min-height:0; height:100%; overflow-y:auto; overflow-x:hidden; border:1px solid #d9e4ef; border-radius:8px; padding:6px; background:#fcfdff; }
    #neighSevereChart { height:auto !important; width:100% !important; }
    #neighborhoodTierDistChart { height:100% !important; width:100% !important; margin-top:0; }
    @media (max-width: 900px) {
      #leafletMap { height: 340px; }
      #dailyChart, #histChart, #thresholdChart { height:220px !important; }
      .weather-panel .panel-body, .neigh-panel .panel-body { height:420px; }
      .map-dist-wrap { height:190px; }
    }

    .insight-list { margin:8px 0 0 16px; }
    .insight-list li { margin-bottom:6px; }
    .section-divider {
      display:flex; align-items:center; gap:10px; margin:10px 2px 6px;
      color:#3b5b7a; font-size:11px; font-weight:700; text-transform:uppercase; letter-spacing:0.04em;
    }
    .section-divider::before, .section-divider::after { content:''; height:1px; background:#b9cde2; flex:1; }
    .section-note { font-size:11px; color:#44566c; margin-bottom:8px; }
    .ml-list { margin:4px 0 0 16px; font-size:11px; color:#44566c; }
    .ml-list li { margin-bottom:5px; }
    .status-chip {
      display:inline-block; font-size:10px; font-weight:700; color:#7a4d00;
      background:#ffe7b8; border:1px solid #f4c067; border-radius:999px; padding:2px 8px; margin-bottom:8px;
    }
    .subpanel-title { margin-top:10px; font-size:11px; font-weight:700; color:#35506a; }
    .map-dist-wrap { position:relative; height:210px; width:100%; }
    .v-stack { display:grid; grid-template-rows:minmax(260px,1.35fr) minmax(180px,1fr); gap:10px; height:100%; }
    .v-stack .panel { height:100%; }
    @media (max-width: 1180px) { .v-stack { grid-template-rows:auto auto; height:auto; } }
  </style>
</head>
<body>
  <div class="header">
    <div class="header-title">Chicago Traffic Crash Risk Dashboard</div>
    <div class="header-subtitle">CMU ML Midterm Project • Rizaldy, Diyouva, Utami</div>
  </div>
  <div class="dashboard">
    <div class="panel" style="margin-bottom:10px;">
      <div class="panel-header"><div class="panel-title">Data Scope</div></div>
      <div class="panel-body" id="scope"></div>
    </div>

    <div class="kpi-grid" id="kpiGrid"></div>

    <div class="section-divider"><span>Descriptive Analysis (EDA and Spatial Context)</span></div>

    <div class="row">
      <div class="col-2">
        <div class="panel map-panel">
          <div class="panel-header"><div class="panel-title">Neighborhood Map (OpenStreetMap) with Filters</div></div>
          <div class="panel-body">
            <div class="filters">
              <label>Risk Tier: <select id="tierFilter"></select></label>
              <span class="selected-chip">Neighborhood polygon: <b id="selectedNeighborhoodText">ALL</b></span>
              <button id="clearNeighborhoodBtn" class="ghost-btn" type="button">Clear polygon filter</button>
            </div>
            <div id="leafletMap"></div>
            <div class="subpanel-title">Selected Polygon Tier Distribution (Severe Rate %)</div>
            <div class="map-dist-wrap"><canvas id="neighborhoodTierDistChart"></canvas></div>
          </div>
        </div>
      </div>
      <div class="col">
        <div class="v-stack">
        <div class="panel">
          <div class="panel-header"><div class="panel-title">Daily Crash Timeline</div></div>
          <div class="panel-body"><div class="filters"><label>Timeline: <select id="timelineGranularity"><option value="day" selected>Daily</option><option value="week">Weekly</option><option value="month">Monthly</option></select></label></div><canvas id="dailyChart" height="220"></canvas></div>
        </div>
        <div class="panel">
          <div class="panel-header"><div class="panel-title">Key EDA Insights</div></div>
          <div class="panel-body"><ul class="insight-list" id="insightList"></ul></div>
        </div>
        </div>
      </div>
    </div>

    <div class="row">
      <div class="col-2"><div class="panel weather-panel"><div class="panel-header"><div class="panel-title">Weather × Lighting Severe Rate (Cross-Filtered Heatmap)</div></div><div class="panel-body"><div id="wlHeatmap"></div></div></div></div>
      <div class="col"><div class="panel neigh-panel"><div class="panel-header"><div class="panel-title">Neighborhood Severe Rate Ranking</div></div><div class="panel-body"><p class="section-note" style="margin-bottom:6px;">Scrollable ranking. Click a bar to filter map and linked charts.</p><div class="neigh-scroll"><canvas id="neighSevereChart"></canvas></div></div></div></div>
    </div>

    <div class="tier-grid" id="tierCards"></div>

    <div class="row">
      <div class="col-2"><div class="panel"><div class="panel-header"><div class="panel-title">Temporal Heatmap (Day × Hour)</div></div><div class="panel-body"><div id="temporalHeatmap"></div></div></div></div>
      <div class="col"><div class="panel"><div class="panel-header"><div class="panel-title">Risk Distribution</div></div><div class="panel-body"><canvas id="histChart" height="220"></canvas></div></div></div>
    </div>

    <div class="row">
      <div class="col"><div class="panel"><div class="panel-header"><div class="panel-title">Condition Breakdown (Severe Rate %)</div></div><div class="panel-body"><div id="condBars" class="small-bars"></div></div></div></div>
      <div class="col-2"><div class="panel"><div class="panel-header"><div class="panel-title">Risk Tier → Recommendation (Sankey)</div></div><div class="panel-body"><svg id="sankey" width="100%" height="280"></svg></div></div></div>
    </div>

    <div class="panel" style="margin-bottom:10px;"><div class="panel-header"><div class="panel-title">Top Condition Profiles</div></div><div class="panel-body table-wrap"><table id="topTable"></table></div></div>

    <div class="row">
      <div class="col"><div class="panel"><div class="panel-header"><div class="panel-title">Policy Threshold Analysis</div></div><div class="panel-body"><canvas id="thresholdChart" height="210"></canvas></div></div></div>
    </div>

    <div class="section-divider"><span>Machine Learning Output (Placeholder)</span></div>

    <div class="row">
      <div class="col"><div class="panel"><div class="panel-header"><div class="panel-title">Classification Output (Notebook 03, TODO)</div></div><div class="panel-body"><span class="status-chip">TODO</span><p class="section-note">Planned model evaluation summary for severe-crash classification.</p><ul class="ml-list"><li><strong>Input:</strong> <code>dataset/cleaned/train_model.csv</code>, <code>dataset/cleaned/test_model.csv</code>, <code>dataset/cleaned/feature_columns.json</code></li><li><strong>Process:</strong> train baseline and tuned models, compare precision/recall/F1/AUC, assess calibration and threshold tradeoff</li><li><strong>Output:</strong> metrics table and plots exported by Notebook 03 for integration into this dashboard</li></ul></div></div></div>
      <div class="col"><div class="panel"><div class="panel-header"><div class="panel-title">Clustering Output (Notebook 04, TODO)</div></div><div class="panel-body"><span class="status-chip">TODO</span><p class="section-note">Planned segment discovery for crash contexts and spatial profiles.</p><ul class="ml-list"><li><strong>Input:</strong> engineered profile/grid features exported from Notebook 02</li><li><strong>Process:</strong> run clustering diagnostics, label cluster behavior, map clusters to intervention themes</li><li><strong>Output:</strong> cluster summary tables and map overlays exported by Notebook 04 for this dashboard</li></ul></div></div></div>
    </div>

    <div class="footer">
      <strong>Key interpretation:</strong>
      <ul>
        <li>Top ranked condition profiles are concentrated in a small subset of zones.</li>
        <li>Night-time and adverse visibility contexts remain overrepresented in severe outcomes.</li>
        <li>Threshold chart supports a transparent tradeoff between intervention scope and severe-incident capture.</li>
      </ul>
    </div>
  </div>

  <script>
    const WL = __WL__;
    const TEMP = __TEMP__;
    const COND = __COND__;
    const SANKEY = __SANKEY__;
    const THRESH = __THRESH__;
    const HIST = __HIST__;
    const TOP = __TOP__;
    const KPI = __KPI__;
    const TIER = __TIER__;
    const DAILY = __DAILY__;
    const DAILY_DETAIL = __DAILY_DETAIL__;
    const WL_DETAIL = __WL_DETAIL__;
    const MAPPTS = __MAP_POINTS__;
    const NEIGH_GEO = __NEIGH_GEO__;
    const INSIGHTS = __INSIGHTS__;
    const NEIGH = __NEIGHBORHOOD__;

    const colors = { PRIORITY:'#dc2626', HIGH:'#ea580c', MODERATE:'#ca8a04', STANDARD:'#16a34a' };
    Chart.register(ChartDataLabels);

    document.getElementById('scope').innerHTML = `Crash-level base: <b>${KPI.total_crashes.toLocaleString()}</b> rows | Severe rate: <b>${KPI.severe_rate_pct.toFixed(2)}%</b> | This dashboard combines descriptive EDA and spatial context before model decisions.`;

    const kpis = [
      ['Total Crashes', KPI.total_crashes.toLocaleString(), 'All crash records (train + test)'],
      ['Severe Rate', KPI.severe_rate_pct.toFixed(2) + '%', 'Crash-level target prevalence'],
      ['High-Risk Zones', KPI.high_risk_zones.toLocaleString(), 'Unique grid cells at rank ≤ 200'],
      ['Max Risk Score', KPI.max_risk_score.toFixed(4), 'Highest profile risk score']
    ];
    document.getElementById('kpiGrid').innerHTML = kpis.map(([label,val,detail]) => `<div class="kpi-card"><div class="kpi-label">${label}</div><div class="kpi-value">${val}</div><div class="kpi-detail">${detail}</div></div>`).join('');

    const insightItems = [
      `Top crash-volume neighborhoods: ${INSIGHTS.top_volume.join(', ')}.`,
      `Top severe-rate neighborhoods (>=100 crashes): ${INSIGHTS.top_severe_rate.join(', ')}.`,
      `Peak daily crash volume reached ${INSIGHTS.peak_crashes.toLocaleString()} on ${INSIGHTS.peak_day}.`,
      `Average daily crashes in full dataset: ${INSIGHTS.daily_avg_crashes.toFixed(1)}.`
    ];
    document.getElementById('insightList').innerHTML = insightItems.map(t => `<li>${t}</li>`).join('');

    const toneRules = [
      { re: /(Data Scope|KPI|Timeline|Threshold)/i, cls: 'tone-blue' },
      { re: /(Map|Neighborhood)/i, cls: 'tone-teal' },
      { re: /(Severe|Risk Distribution|Sankey)/i, cls: 'tone-red' },
      { re: /(Weather|Temporal|Condition)/i, cls: 'tone-orange' },
      { re: /(Top Condition Profiles|Insights)/i, cls: 'tone-purple' },
      { re: /(Placeholder|TODO)/i, cls: 'tone-grey' }
    ];
    document.querySelectorAll('.panel').forEach(panel => {
      const title = panel.querySelector('.panel-title')?.textContent || '';
      const rule = toneRules.find(r => r.re.test(title));
      panel.classList.add(rule ? rule.cls : 'tone-blue');
    });

    function withAlpha(hex, alpha){
      const h = (hex || '').replace('#','');
      if (h.length !== 6) return hex;
      const r = parseInt(h.slice(0,2), 16);
      const g = parseInt(h.slice(2,4), 16);
      const b = parseInt(h.slice(4,6), 16);
      return `rgba(${r}, ${g}, ${b}, ${alpha})`;
    }

    let selectedNeighborhood = 'ALL';

    function updateSelectedNeighborhoodLabel(){
      const el = document.getElementById('selectedNeighborhoodText');
      if (el) el.textContent = selectedNeighborhood || 'ALL';
    }

    function applyNeighborhoodCrossFilter(community, tier){
      if (typeof tierFilter !== 'undefined' && tierFilter && tier) {
        tierFilter.value = tier;
      }
      selectedNeighborhood = community || 'ALL';
      updateSelectedNeighborhoodLabel();
      if (typeof renderMapPoints === 'function') {
        renderMapPoints();
      }
      if (typeof focusNeighborhood === 'function') {
        focusNeighborhood(selectedNeighborhood, true);
      }
      if (typeof renderNeighborhoodTierDist === 'function') {
        renderNeighborhoodTierDist();
      }
      if (typeof renderTimeline === 'function') {
        renderTimeline();
      }
      if (typeof renderWeatherHeatmap === 'function') {
        renderWeatherHeatmap();
      }
    }

    const neighborhoodRows = Array.isArray(NEIGH) ? NEIGH : [];
    const neighCanvas = document.getElementById('neighSevereChart');
    const neighScroll = document.querySelector('.neigh-scroll');
    const severeVals = neighborhoodRows.map(d => Number(d.severe_rate_pct) || 0);
    const severeMin = severeVals.length ? Math.min(...severeVals) : 0;
    const severeMax = severeVals.length ? Math.max(...severeVals) : 1;
    const severeScale = d3.scaleSequential().domain([severeMin, severeMax]).interpolator(d3.interpolatePuBu);

    function setNeighCanvasSize() {
      if (!neighCanvas) return;
      neighCanvas.height = Math.max(360, neighborhoodRows.length * 22);
      if (neighScroll) {
        neighCanvas.width = Math.max(360, Math.floor(neighScroll.clientWidth - 12));
      }
    }

    setNeighCanvasSize();

    const neighSevereChart = new Chart(neighCanvas, {
      type: 'bar',
      data: {
        labels: neighborhoodRows.map(d => d.community),
        datasets: [{
          label: 'Severe rate (%)',
          data: severeVals,
          backgroundColor: severeVals.map(v => severeScale(v)),
          borderColor: severeVals.map(v => {
            const c = d3.color(severeScale(v));
            return c ? c.darker(0.8).formatHex() : '#4e79a7';
          }),
          borderWidth: 1,
          barThickness: 13,
          maxBarThickness: 15,
          categoryPercentage: 0.9,
          barPercentage: 0.95
        }]
      },
      options: {
        indexAxis: 'y',
        responsive: false,
        maintainAspectRatio: false,
        animation: false,
        plugins: { datalabels: { display: false }, legend: { position: 'top' } },
        scales: {
          x: { beginAtZero: true, title: { display: true, text: 'Severe rate (%)' } },
          y: { ticks: { autoSkip: false, maxRotation: 0, minRotation: 0, font: { size: 10 } } }
        },
        onClick: (evt, _elements, chart) => {
          const hit = chart.getElementsAtEventForMode(evt, 'nearest', { intersect: true }, true);
          if (!hit.length) return;
          const i = hit[0].index;
          const row = neighborhoodRows[i];
          if (!row || !row.community) return;
          applyNeighborhoodCrossFilter(row.community, row.dominant_tier || 'ALL');
        }
      }
    });

    let neighResizeTimer = null;
    window.addEventListener('resize', () => {
      if (neighResizeTimer) clearTimeout(neighResizeTimer);
      neighResizeTimer = setTimeout(() => {
        setNeighCanvasSize();
        if (neighSevereChart) neighSevereChart.resize();
      }, 120);
    });

    const tierOrder = ['PRIORITY', 'HIGH', 'MODERATE', 'STANDARD'];
    const tierDistChart = new Chart(document.getElementById('neighborhoodTierDistChart'), {
      type: 'bar',
      data: {
        labels: tierOrder,
        datasets: [{
          label: 'Severe rate (%) in selected polygon',
          data: [0, 0, 0, 0],
          backgroundColor: tierOrder.map(t => withAlpha(colors[t], 0.35)),
          borderColor: tierOrder.map(t => colors[t]),
          borderWidth: 1.1
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: {
          datalabels: { display: false },
          legend: { display: false },
          title: { display: true, text: 'Neighborhood: ALL' }
        },
        scales: {
          x: { ticks: { autoSkip: false, maxRotation: 0, minRotation: 0 } },
          y: { beginAtZero: true, suggestedMax: 15, title: { display: true, text: 'Severe rate (%)' } }
        }
      }
    });

    function renderNeighborhoodTierDist(){
      const filtered = MAPPTS.filter(p => selectedNeighborhood === 'ALL' || p.n === selectedNeighborhood);
      const rates = tierOrder.map(t => {
        const rows = filtered.filter(p => p.t === t);
        const crashSum = rows.reduce((acc, p) => acc + (Number(p.c) || 0), 0);
        const severeApprox = rows.reduce((acc, p) => acc + ((Number(p.c) || 0) * (Number(p.s) || 0) / 100.0), 0);
        return crashSum > 0 ? (100 * severeApprox / crashSum) : 0;
      });
      tierDistChart.data.datasets[0].data = rates;
      tierDistChart.options.plugins.title.text = `Neighborhood: ${selectedNeighborhood}`;
      tierDistChart.update();
      updateSelectedNeighborhoodLabel();
    }

    function aggregateTimeline(rows, granularity) {
      const bucket = new Map();
      rows.forEach(row => {
        const dt = new Date(row.d + 'T00:00:00');
        let key;
        if (granularity === 'week') {
          const day = dt.getDay();
          const mondayOffset = day === 0 ? -6 : (1 - day);
          const weekStart = new Date(dt);
          weekStart.setDate(dt.getDate() + mondayOffset);
          key = weekStart.toISOString().slice(0, 10);
        } else if (granularity === 'month') {
          key = `${dt.getFullYear()}-${String(dt.getMonth() + 1).padStart(2, '0')}`;
        } else {
          key = row.d;
        }

        if (!bucket.has(key)) bucket.set(key, {c: 0, s: 0});
        const acc = bucket.get(key);
        acc.c += Number(row.c) || 0;
        acc.s += Number(row.s) || 0;
      });

      return Array.from(bucket.entries())
        .sort((a, b) => a[0].localeCompare(b[0]))
        .map(([d, v]) => {
          const c = Number(v.c) || 0;
          const s = Number(v.s) || 0;
          const sr = c > 0 ? (100 * s / c) : 0;
          return { d, c, s, sr };
        });
    }

    function activeTierValue(){
      const el = document.getElementById('tierFilter');
      const v = el ? el.value : '';
      return v ? v : 'ALL';
    }

    function getTimelineBaseRows(){
      const t = activeTierValue();
      const isGlobal = selectedNeighborhood === 'ALL' && t === 'ALL';
      if (isGlobal) return DAILY;
      return DAILY_DETAIL.filter(r => (selectedNeighborhood === 'ALL' || r.n === selectedNeighborhood) && (t === 'ALL' || r.t === t));
    }

    const timelineGranularity = document.getElementById('timelineGranularity');
    let dailyChart = null;
    function renderTimeline() {
      const granularity = timelineGranularity.value;
      const rows = aggregateTimeline(getTimelineBaseRows(), granularity);
      if (dailyChart) dailyChart.destroy();
      dailyChart = new Chart(document.getElementById('dailyChart'), {
        type:'line',
        data:{
          labels: rows.map(d => d.d),
          datasets:[
            {label:'Crash count', data: rows.map(d=>d.c), borderColor:'#1d4ed8', pointRadius:0, tension:0.2, fill:false, yAxisID:'y'},
            {label:'Severe rate (%)', data: rows.map(d=>d.sr), borderColor:'#dc2626', pointRadius:0, tension:0.2, fill:false, yAxisID:'y1'}
          ]
        },
        options:{
          responsive:true,
          maintainAspectRatio:false,
          interaction:{ mode:'index', intersect:false },
          plugins:{ datalabels:{display:false}, legend:{position:'top'} },
          scales:{
            x:{ticks:{maxTicksLimit:10}},
            y:{beginAtZero:true, title:{display:true, text:'Crash count'}},
            y1:{
              beginAtZero:true,
              position:'right',
              grid:{drawOnChartArea:false},
              title:{display:true, text:'Severe rate (%)'},
              ticks:{ callback:(value)=> `${Number(value).toFixed(1)}%` }
            }
          }
        }
      });
    }
    timelineGranularity.addEventListener('change', renderTimeline);
    renderTimeline();

    const map = L.map('leafletMap').setView([41.8781, -87.6298], 10);
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
      attribution: '&copy; OpenStreetMap contributors &copy; CARTO',
      maxZoom: 18
    }).addTo(map);

    function choroplethColor(v){
      return v > 12000 ? '#7f0000' : v > 8000 ? '#b30000' : v > 5000 ? '#d7301f' : v > 3000 ? '#ef6548' : v > 1500 ? '#fc8d59' : v > 500 ? '#fdbb84' : '#fee8c8';
    }

    const neighborhoodLayerLookup = {};
    const neighborhoodGeoLayer = L.geoJSON(NEIGH_GEO, {
      style: f => ({ color:'#666', weight:1, fillOpacity:0.5, fillColor: choroplethColor(f.properties.crash_count || 0) }),
      onEachFeature: (f, layer) => {
        const p = f.properties;
        const name = p.community;
        neighborhoodLayerLookup[name] = layer;
        layer.bindTooltip(`${p.community}<br>Crashes: ${Number(p.crash_count||0).toLocaleString()}<br>Severe rate: ${Number(p.severe_rate_pct||0).toFixed(2)}%<br>Avg risk: ${Number(p.avg_risk_score||0).toFixed(4)}`);
        layer.on('click', () => {
          selectedNeighborhood = name;
          updateSelectedNeighborhoodLabel();
          renderMapPoints();
          focusNeighborhood(name, true);
          renderNeighborhoodTierDist();
          renderTimeline();
          renderWeatherHeatmap();
        });
      }
    }).addTo(map);

    function resetNeighborhoodStyles(){
      Object.values(neighborhoodLayerLookup).forEach(layer => neighborhoodGeoLayer.resetStyle(layer));
    }

    function focusNeighborhood(name, shouldFit){
      resetNeighborhoodStyles();
      if (!name || name === 'ALL' || !neighborhoodLayerLookup[name]) return;
      const layer = neighborhoodLayerLookup[name];
      layer.setStyle({ color:'#1d3557', weight:3, fillOpacity:0.7 });
      if (layer.bringToFront) layer.bringToFront();
      if (shouldFit && layer.getBounds) {
        map.fitBounds(layer.getBounds(), { padding: [18, 18], maxZoom: 12 });
      }
    }

    const tierFilter = document.getElementById('tierFilter');
    tierFilter.innerHTML = ['ALL','PRIORITY','HIGH','MODERATE','STANDARD'].map(t => `<option value="${t}">${t}</option>`).join('');

    const pointsLayer = L.layerGroup().addTo(map);
    const osmRoadCache = new Map();

    async function reverseRoadOSM(lat, lon){
      const key = `${Number(lat).toFixed(4)},${Number(lon).toFixed(4)}`;
      if (osmRoadCache.has(key)) return osmRoadCache.get(key);
      const url = `https://nominatim.openstreetmap.org/reverse?format=jsonv2&lat=${encodeURIComponent(lat)}&lon=${encodeURIComponent(lon)}&zoom=18&addressdetails=1`;
      try {
        const res = await fetch(url, { headers: { 'Accept': 'application/json' } });
        if (!res.ok) throw new Error(`status ${res.status}`);
        const data = await res.json();
        const adr = data.address || {};
        const road = adr.road || adr.pedestrian || adr.footway || adr.path || adr.cycleway || adr.highway || 'Unavailable';
        osmRoadCache.set(key, road);
        return road;
      } catch (_err) {
        osmRoadCache.set(key, 'Unavailable');
        return 'Unavailable';
      }
    }

    function tierRadius(c){ return Math.max(2, Math.min(8, Math.sqrt(c||1))); }
    function popupHTML(p, osmRoad){
      const road = (osmRoad || String(p.tw || 'UNKNOWN').replace(/_/g, ' '));
      return `Neighborhood: ${p.n}<br>Tier: ${p.t}<br>Crashes: ${Number(p.c).toLocaleString()}<br>Severe rate: ${Number(p.s).toFixed(2)}%<br>Road context: ${String(p.tw || 'UNKNOWN').replace(/_/g, ' ')}<br>OSM nearest road: ${road}<br>Approx point: (${Number(p.la).toFixed(4)}, ${Number(p.lo).toFixed(4)})`;
    }

    function renderMapPoints(){
      pointsLayer.clearLayers();
      const t = tierFilter.value;
      MAPPTS
        .filter(p => (t === 'ALL' || p.t === t) && (selectedNeighborhood === 'ALL' || p.n === selectedNeighborhood))
        .forEach(p => {
          const marker = L.circleMarker([p.la, p.lo], {
            radius: tierRadius(p.c),
            color: colors[p.t] || '#555',
            fillColor: colors[p.t] || '#555',
            fillOpacity: 0.55,
            weight: 0.8
          })
          .bindPopup(popupHTML(p, null))
          .bindTooltip(String(p.tw || 'UNKNOWN').replace(/_/g, ' '), { direction: 'top', opacity: 0.85 });

          marker.on('popupopen', async () => {
            const osmRoad = await reverseRoadOSM(p.la, p.lo);
            marker.setPopupContent(popupHTML(p, osmRoad));
          });

          marker.addTo(pointsLayer);
        });
    }

    tierFilter.addEventListener('change', () => {
      renderMapPoints();
      focusNeighborhood(selectedNeighborhood, false);
      renderNeighborhoodTierDist();
      renderTimeline();
      renderWeatherHeatmap();
    });

    const clearNeighborhoodBtn = document.getElementById('clearNeighborhoodBtn');
    if (clearNeighborhoodBtn) {
      clearNeighborhoodBtn.addEventListener('click', () => {
        selectedNeighborhood = 'ALL';
        updateSelectedNeighborhoodLabel();
        renderMapPoints();
        focusNeighborhood('ALL', false);
        renderNeighborhoodTierDist();
        renderTimeline();
        renderWeatherHeatmap();
      });
    }

    updateSelectedNeighborhoodLabel();
    renderMapPoints();
    focusNeighborhood('ALL', false);
    renderNeighborhoodTierDist();


    const actionTag = { PRIORITY:'IMMEDIATE ACTION', HIGH:'SAFETY AUDIT', MODERATE:'MONITOR', STANDARD:'ROUTINE' };
    document.getElementById('tierCards').innerHTML = TIER.map(r => {
      const t = r.tier;
      return `<div class="tier-card" style="border-top:4px solid ${colors[t]};"><h4>${t}</h4><div class="big">${Number(r.cell_count).toLocaleString()} cells</div><div>Avg risk: ${Number(r.avg_risk).toFixed(4)}</div><div>Crashes: ${Number(r.crash_count).toLocaleString()}</div><span class="tag" style="background:${colors[t]}">${actionTag[t]}</span></div>`;
    }).join('');




    function colorScale(v, min, max) {
      if (v === null || Number.isNaN(v)) return '#f6f6f6';
      const ratio = (v - min) / Math.max(1e-9, (max - min));
      const r = Math.round(255 - ratio * 55);
      const g = Math.round(250 - ratio * 180);
      const b = Math.round(220 - ratio * 200);
      return `rgb(${r},${g},${b})`;
    }

    function getFilteredWLRows(){
      const t = activeTierValue();
      return WL_DETAIL.filter(r => (selectedNeighborhood === 'ALL' || r.n === selectedNeighborhood) && (t === 'ALL' || r.t === t));
    }

    function renderWeatherHeatmap(){
      const wrap = document.getElementById('wlHeatmap');
      const rows = getFilteredWLRows();
      if (!rows.length) {
        wrap.innerHTML = '<div class="section-note">No weather-lighting rows for current polygon/tier filter.</div>';
        return;
      }

      const keyMap = new Map();
      rows.forEach(r => {
        const key = `${r.w}|||${r.l}`;
        if (!keyMap.has(key)) keyMap.set(key, {w: r.w, l: r.l, c: 0, s: 0});
        const acc = keyMap.get(key);
        acc.c += Number(r.c) || 0;
        acc.s += Number(r.s) || 0;
      });
      const agg = Array.from(keyMap.values()).map(d => ({
        w: d.w,
        l: d.l,
        sr: d.c > 0 ? (100 * d.s / d.c) : 0
      }));

      const weather = Array.from(new Set(agg.map(d => d.w))).sort();
      const lighting = Array.from(new Set(agg.map(d => d.l))).sort();
      const values = agg.map(d => d.sr);
      const vmin = Math.min(...values), vmax = Math.max(...values);

      wrap.innerHTML = '';
      const width = Math.max(360, wrap.clientWidth || 360);
      const margin = { top: 22, right: 8, bottom: 66, left: 120 };
      const cellW = Math.max(58, Math.floor((width - margin.left - margin.right) / Math.max(1, lighting.length)));
      const cellH = 26;
      const height = margin.top + cellH * weather.length + margin.bottom;

      const svg = d3.select(wrap).append('svg').attr('width', width).attr('height', height);
      const g = svg.append('g').attr('transform', `translate(${margin.left},${margin.top})`);
      const color = d3.scaleSequential().domain([vmin, vmax]).interpolator(d3.interpolateYlOrRd);

      const lookup = new Map(agg.map(d => [`${d.w}|||${d.l}`, d.sr]));
      weather.forEach((w, i) => {
        lighting.forEach((l, j) => {
          const sr = lookup.get(`${w}|||${l}`) ?? 0;
          g.append('rect')
            .attr('x', j * cellW)
            .attr('y', i * cellH)
            .attr('width', cellW - 1)
            .attr('height', cellH - 1)
            .attr('fill', color(sr));
          g.append('text')
            .attr('x', j * cellW + (cellW / 2))
            .attr('y', i * cellH + (cellH / 2) + 3)
            .attr('text-anchor', 'middle')
            .attr('font-size', '10px')
            .attr('fill', sr > (vmin + vmax) / 2 ? '#fff' : '#1f2937')
            .text(`${sr.toFixed(1)}%`);
        });
      });

      weather.forEach((w, i) => {
        g.append('text')
          .attr('x', -8)
          .attr('y', i * cellH + (cellH / 2) + 3)
          .attr('text-anchor', 'end')
          .attr('font-size', '10px')
          .text(w);
      });

      lighting.forEach((l, j) => {
        g.append('text')
          .attr('x', j * cellW + (cellW / 2))
          .attr('y', cellH * weather.length + 14)
          .attr('text-anchor', 'end')
          .attr('transform', `rotate(-35 ${j * cellW + (cellW / 2)} ${cellH * weather.length + 14})`)
          .attr('font-size', '10px')
          .text(l);
      });

      svg.append('text').attr('x', margin.left).attr('y', 12).attr('font-size', '11px').attr('font-weight', '700')
        .text(`Severe rate (%) | Neighborhood: ${selectedNeighborhood} | Tier: ${activeTierValue()}`);
    }

    renderWeatherHeatmap();

    const tempVals = TEMP.vals.flat().filter(v => v !== null);
    const tMin = Math.min(...tempVals), tMax = Math.max(...tempVals);
    const grid = document.getElementById('temporalHeatmap');
    grid.className = 'heatgrid';
    grid.style.gridTemplateColumns = `70px repeat(${TEMP.hours.length}, 1fr)`;
    grid.innerHTML = `<div class="cell"></div>` + TEMP.hours.map(h => `<div class="cell">${h}</div>`).join('');
    TEMP.days.forEach((d, i) => {
      grid.innerHTML += `<div class="cell"><b>${d}</b></div>`;
      TEMP.hours.forEach((_, j) => {
        const v = TEMP.vals[i][j];
        grid.innerHTML += `<div class="cell" style="background:${colorScale(v, tMin, tMax)}">${v===null?'':v.toFixed(1)}</div>`;
      });
    });

    new Chart(document.getElementById('histChart'), {
      type:'line',
      data: {
        datasets:[{
          label:'Density',
          data:HIST.map(h => ({x:h.x, y:h.density})),
          borderColor:'#1d4ed8',
          backgroundColor:'rgba(147,197,253,0.35)',
          fill:true,
          pointRadius:0,
          tension:0.25
        }]
      },
      options: {
        responsive:true,
        maintainAspectRatio:false,
        plugins: { datalabels: { display:false }, legend:{position:'top'}},
        scales: {
          x: { type:'linear', title:{display:true, text:'Risk score'} },
          y: { beginAtZero:true, title:{display:true, text:'Density'} }
        }
      }
    });

    const condColor = { weather:'#4e79a7', lighting:'#f28e2b', time:'#e15759', speed:'#76b7b2' };
    const condOrder = ['weather', 'lighting', 'time', 'speed'];

    const condHtml = condOrder.map(grp => {
      const rows = (COND[grp] || []).slice().sort((a, b) => (Number(b.severe_rate_pct) || 0) - (Number(a.severe_rate_pct) || 0)).slice(0, 7);
      const maxV = Math.max(...rows.map(r => Number(r.severe_rate_pct) || 0), 1e-9);
      const body = rows.map((r, idx) => {
        const label = (r.weather_bucket || r.lighting_bucket || r.time_bucket || r.speed_bucket || 'UNKNOWN');
        const val = Number(r.severe_rate_pct) || 0;
        return `<div class="grp"><div class="lbl">${idx + 1}. ${label} • ${val.toFixed(2)}%</div><div class="bar"><div class="fill" style="width:${(val/maxV*100).toFixed(1)}%; background:${condColor[grp] || '#4e79a7'}"></div></div></div>`;
      }).join('');
      return `<div class="cond-col"><div class="cond-head" style="color:${condColor[grp] || '#4e79a7'}">${grp}</div>${body}</div>`;
    }).join('');

    document.getElementById('condBars').innerHTML = `<div class="cond-grid">${condHtml}</div>`;

    const headers = ['rank','grid_lat','grid_lon','community','weather_bucket','lighting_bucket','time_bucket','speed_bucket','crash_count','severe_rate_pct','risk_score','recommendation'];
    let tHtml = '<thead><tr>' + headers.map(h => `<th>${h}</th>`).join('') + '</tr></thead><tbody>';
    TOP.forEach(r => {
      tHtml += '<tr>' + headers.map(h => `<td>${typeof r[h]==='number' ? (h.includes('rate') ? r[h].toFixed(2) : Number(r[h]).toFixed(4).replace(/\.0000$/,'')) : r[h]}</td>`).join('') + '</tr>';
    });
    tHtml += '</tbody>';
    document.getElementById('topTable').innerHTML = tHtml;

    new Chart(document.getElementById('thresholdChart'), {
      type:'line',
      data: { labels: THRESH.map(d=>d.rank_cutoff), datasets:[
        { label:'Zones captured', data:THRESH.map(d=>d.zone_count), borderColor:'#1d4ed8', backgroundColor:'#1d4ed822', yAxisID:'y' },
        { label:'Severe capture (%)', data:THRESH.map(d=>d.severe_capture_pct), borderColor:'#dc2626', backgroundColor:'#dc262622', yAxisID:'y1' }
      ]},
      options: { responsive:true, maintainAspectRatio:false, interaction:{ mode:'index', intersect:false }, plugins: { datalabels: { display:false }, legend:{position:'top'}}, scales: { y: { position:'left', beginAtZero:true }, y1: { position:'right', beginAtZero:true, grid: { drawOnChartArea:false } } } }
    });




    const svg = d3.select('#sankey');
    const width = svg.node().getBoundingClientRect().width || 700;
    const height = 280;
    svg.attr('viewBox', `0 0 ${width} ${height}`);
    const nodes = Array.from(new Set(SANKEY.flatMap(d => [d.tier, d.recommendation]))).map(name => ({name}));
    const nodeIndex = new Map(nodes.map((n, i) => [n.name, i]));
    const links = SANKEY.map(d => ({ source: nodeIndex.get(d.tier), target: nodeIndex.get(d.recommendation), value: d.value }));

    const sankey = d3.sankey().nodeWidth(12).nodePadding(10).extent([[1, 1], [width - 1, height - 6]]);
    const graph = sankey({ nodes: nodes.map(d => Object.assign({}, d)), links: links.map(d => Object.assign({}, d)) });

    svg.append('g').selectAll('rect')
      .data(graph.nodes)
      .join('rect')
      .attr('x', d => d.x0).attr('y', d => d.y0)
      .attr('height', d => Math.max(1, d.y1 - d.y0)).attr('width', d => d.x1 - d.x0)
      .attr('fill', d => colors[d.name] || '#6366f1').attr('opacity', 0.9);

    svg.append('g').attr('fill', 'none').selectAll('path')
      .data(graph.links)
      .join('path')
      .attr('d', d3.sankeyLinkHorizontal())
      .attr('stroke', '#9ca3af')
      .attr('stroke-width', d => Math.max(1, d.width))
      .attr('opacity', 0.35);

    svg.append('g').style('font', '10px sans-serif').selectAll('text')
      .data(graph.nodes)
      .join('text')
      .attr('x', d => d.x0 < width / 2 ? d.x1 + 6 : d.x0 - 6)
      .attr('y', d => (d.y0 + d.y1) / 2)
      .attr('dy', '0.35em')
      .attr('text-anchor', d => d.x0 < width / 2 ? 'start' : 'end')
      .text(d => d.name);
  </script>
</body>
</html>
'''

html = (
    template
    .replace('__WL__', wl_blob)
    .replace('__TEMP__', temporal_blob)
    .replace('__COND__', condition_blob)
    .replace('__SANKEY__', sankey_blob)
    .replace('__THRESH__', threshold_blob)
    .replace('__HIST__', hist_blob)
    .replace('__TOP__', top_blob)
    .replace('__KPI__', kpi_blob)
    .replace('__TIER__', tier_blob)
    .replace('__DAILY__', daily_blob)
    .replace('__DAILY_DETAIL__', daily_detail_blob)
    .replace('__WL_DETAIL__', wl_detail_blob)
    .replace('__MAP_POINTS__', map_points_blob)
    .replace('__NEIGH_GEO__', neigh_geo_blob)
    .replace('__INSIGHTS__', insight_blob)
    .replace('__NEIGHBORHOOD__', neighborhood_blob)
)

html_path = IMAGE_DIR / 'chicago_crash_risk_dashboard.html'
html_path.write_text(html, encoding='utf-8')

print(f'Wrote HTML dashboard: {html_path}')
print(f'HTML size: {html_path.stat().st_size / 1024:.2f} KB')


In [ ]:
# quick artifact check
required_pngs = [
    IMAGE_DIR / '01_weather_lighting_heatmap.png',
    IMAGE_DIR / '02_temporal_heatmap.png',
    IMAGE_DIR / '03_risk_distribution.png',
    IMAGE_DIR / '04_policy_threshold.png',
]

for p in required_pngs + [IMAGE_DIR / 'chicago_crash_risk_dashboard.html']:
    print(f'{p.name:36s} -> {p.exists()}')

## 5) CSV Exports for Policy and Dashboard Continuity

In [ ]:
grid_reco = (
    risk_scores
    .sort_values('rank')
    .groupby(['grid_lat', 'grid_lon'], as_index=False)
    .first()[['grid_lat', 'grid_lon', 'recommendation']]
)

policy_priority_cells = (
    grid_cells
    .merge(grid_reco, on=['grid_lat', 'grid_lon'], how='left')
    .copy()
)

policy_priority_cells['priority_score'] = (
    policy_priority_cells['risk_score'].rank(pct=True)
)

policy_out = policy_priority_cells[[
    'grid_lat', 'grid_lon', 'risk_score', 'priority_score',
    'cluster_kmeans', 'cluster_dbscan', 'is_high_risk_cluster', 'recommendation'
]].sort_values('risk_score', ascending=False)

policy_path = CLEAN_DIR / 'policy_priority_cells.csv'
policy_out.to_csv(policy_path, index=False)
policy_out.head()


In [ ]:
dashboard_table = (
    policy_priority_cells[[
        'grid_lat', 'grid_lon', 'risk_score', 'crash_count', 'severe_rate',
        'cluster_kmeans', 'cluster_dbscan', 'is_high_risk_cluster', 'recommendation'
    ]]
    .sort_values('risk_score', ascending=False)
)

dashboard_path = CLEAN_DIR / 'dashboard_risk_cluster_table.csv'
dashboard_table.to_csv(dashboard_path, index=False)

dashboard_table.head()


## 6) Handoff Summary

In [ ]:
outputs = [
    IMAGE_DIR / 'chicago_crash_risk_dashboard.html',
    IMAGE_DIR / '01_weather_lighting_heatmap.png',
    IMAGE_DIR / '02_temporal_heatmap.png',
    IMAGE_DIR / '03_risk_distribution.png',
    IMAGE_DIR / '04_policy_threshold.png',
    CLEAN_DIR / 'policy_priority_cells.csv',
    CLEAN_DIR / 'dashboard_risk_cluster_table.csv',
]

summary = pd.DataFrame([
    {'path': str(p), 'exists': p.exists(), 'size_kb': round(p.stat().st_size / 1024, 2) if p.exists() else np.nan}
    for p in outputs
])

display(summary)
print('\npolicy_priority_cells columns:', list(policy_out.columns))
print('dashboard_risk_cluster_table columns:', list(dashboard_table.columns))


## 6) Key Takeaways & Policy Recommendations

- Severe crashes remain a minority class, so the project correctly treats classification as an imbalance-sensitive problem rather than a raw-accuracy exercise.
- Risk is not explained by frequency alone. The feature engineering, EDA, and condition-level ranking consistently point to context, roadway environment, and time windows as the more useful policy lens.
- Spatial aggregation, clustering, and dashboard outputs together create an operational handoff: CDOT can prioritize high-risk cells, inspect neighborhood context, and align interventions with specific risk tiers.


In [ ]:
artifact_paths = [
    DATA_ROOT / 'cleaned_data' / 'train.csv',
    DATA_ROOT / 'cleaned_data' / 'test.csv',
    DATA_ROOT / 'cleaned_data' / 'train_model.csv',
    DATA_ROOT / 'cleaned_data' / 'test_model.csv',
    DATA_ROOT / 'cleaned_data' / 'risk_scores.csv',
    DATA_ROOT / 'cleaned_data' / 'cluster_labels.csv',
    DATA_ROOT / 'cleaned_data' / 'policy_priority_cells.csv',
    DATA_ROOT / 'cleaned_data' / 'dashboard_risk_cluster_table.csv',
    MAIN_IMAGE_DIR / 'chicago_crash_risk_dashboard.html',
]
artifact_summary = pd.DataFrame([
    {'path': str(p), 'exists': p.exists(), 'size_kb': round(p.stat().st_size / 1024, 2) if p.exists() else np.nan}
    for p in artifact_paths
])
display(artifact_summary)
